In [67]:
# ============================================================
# MODULE 5: FEATURE ENGINEERING
# NYC Taxi Trip Duration & Congestion Pricing ML Project
# ============================================================
# Input: Cleaned datasets from Module 4
#   - cleaned_model1_duration.parquet (2,721,501 rows × 22 cols)
#   - cleaned_model2_congestion.parquet (2,412,426 rows × 22 cols)
# Output: Feature-engineered datasets for modeling
#   - fe_model1_duration.parquet
#   - fe_model2_congestion.parquet
# ============================================================
# Feature Engineering Assignments:
#   — Temporal/Time Features (11 features)
#   — Location/Spatial Features (11 features)
#   — Trip Characteristics & Derived Metrics (10 features)
#   — Categorical Encodings & Interaction Features (6 features)
# ============================================================

In [68]:
# ============================================================
# STEP 0: IMPORT LIBRARIES & LOAD CLEANED DATA
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print("=" * 70)
print("MODULE 5: FEATURE ENGINEERING")
print("=" * 70)

MODULE 5: FEATURE ENGINEERING


In [69]:
# ============================================================
# LOAD BOTH CLEANED DATASETS
# ============================================================
# We load both models' datasets because:
# - Model 1 (Duration Regression): 2,721,501 rows — ALL January trips
# - Model 2 (Congestion Classification): 2,412,426 rows — Post-Jan-5 only
# Feature engineering is applied to BOTH datasets in parallel
# ============================================================

# File paths
model1_path = r"C:\Users\hunda\OneDrive\Desktop\Machine Learning Project\processed\cleaned_model1_duration.parquet"
model2_path = r"C:\Users\hunda\OneDrive\Desktop\Machine Learning Project\processed\cleaned_model2_congestion.parquet"

# Load datasets
df_m1 = pd.read_parquet(model1_path)
df_m2 = pd.read_parquet(model2_path)

print(f"\n✅ Model 1 (Duration) loaded: {df_m1.shape[0]:,} rows × {df_m1.shape[1]} columns")
print(f"✅ Model 2 (Congestion) loaded: {df_m2.shape[0]:,} rows × {df_m2.shape[1]} columns")



✅ Model 1 (Duration) loaded: 2,721,501 rows × 22 columns
✅ Model 2 (Congestion) loaded: 2,412,426 rows × 22 columns


In [70]:
# ============================================================
# VERIFY STARTING POINT
# ============================================================
# Quick sanity check — confirm datasets are clean and match
# what we expect from Module 4
# ============================================================

print("\n" + "=" * 70)
print("STARTING POINT VERIFICATION")
print("=" * 70)

# Model 1 checks
print("\n--- Model 1 (Duration Regression) ---")
print(f"Nulls: {df_m1.isnull().sum().sum()}")
print(f"Duration range: {df_m1['trip_duration_min'].min():.2f} to {df_m1['trip_duration_min'].max():.2f} min")
print(f"Columns: {list(df_m1.columns)}")

# Model 2 checks
print("\n--- Model 2 (Congestion Classification) ---")
print(f"Nulls: {df_m2.isnull().sum().sum()}")
print(f"Target distribution:\n{df_m2['has_congestion_fee'].value_counts()}")
print(f"Earliest pickup: {df_m2['tpep_pickup_datetime'].min()}")


STARTING POINT VERIFICATION

--- Model 1 (Duration Regression) ---
Nulls: 0
Duration range: 1.00 to 179.03 min
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee', 'trip_duration_min', 'has_congestion_fee']

--- Model 2 (Congestion Classification) ---
Nulls: 0
Target distribution:
has_congestion_fee
1    1798485
0     613941
Name: count, dtype: int64
Earliest pickup: 2025-01-05 00:00:00


In [71]:
# ============================================================
# ============================================================
#
#   PHASE 1: TEMPORAL FEATURES
#   11 features extracted from pickup datetime
#
# ============================================================
# ============================================================
# WHY TEMPORAL FEATURES MATTER:
# Think of yourself as Maria (taxi driver) sitting in her cab.
# The TIME she picks someone up tells her a LOT:
# - 8 AM Monday = rush hour commuters, heavy traffic, longer trips
# - 2 PM Sunday = leisure travelers, lighter traffic, shorter trips
# - 11 PM Friday = nightlife crowd, different patterns entirely
#
# By extracting time components, we let the ML model learn these
# patterns that Maria knows from experience.
# ============================================================


# ============================================================
# FEATURE 1: pickup_hour (0-23)
# ============================================================
# Extracts the hour from pickup timestamp
# Example: 2025-01-15 08:30:00 → 8
#          2025-01-15 22:15:00 → 22
# WHY: Traffic patterns change dramatically by hour
#      Rush hour (8AM) ≠ Late night (2AM)
# ============================================================

df_m1['pickup_hour'] = df_m1['tpep_pickup_datetime'].dt.hour
df_m2['pickup_hour'] = df_m2['tpep_pickup_datetime'].dt.hour

print("\n" + "=" * 70)
print("PHASE 1: TEMPORAL FEATURES")
print("=" * 70)
print("\n✅ Feature 1: pickup_hour")
print(f"   Range: {df_m1['pickup_hour'].min()} to {df_m1['pickup_hour'].max()}")
print(f"   Most common hour: {df_m1['pickup_hour'].mode()[0]}:00")


PHASE 1: TEMPORAL FEATURES

✅ Feature 1: pickup_hour
   Range: 0 to 23
   Most common hour: 17:00


In [72]:
# ============================================================
# FEATURE 2: pickup_day_of_week (0=Monday ... 6=Sunday)
# ============================================================
# Extracts which day of the week the trip started
# 0=Monday, 1=Tuesday, 2=Wednesday ... 6=Sunday
# WHY: Weekday commute traffic ≠ weekend leisure traffic
#      Tuesday patterns are very different from Sunday
# ============================================================

df_m1['pickup_day_of_week'] = df_m1['tpep_pickup_datetime'].dt.dayofweek
df_m2['pickup_day_of_week'] = df_m2['tpep_pickup_datetime'].dt.dayofweek

print(f"\n✅ Feature 2: pickup_day_of_week")
print(f"   Distribution:\n{df_m1['pickup_day_of_week'].value_counts().sort_index()}")


✅ Feature 2: pickup_day_of_week
   Distribution:
pickup_day_of_week
0    292005
1    362366
2    452298
3    482930
4    456977
5    375226
6    299699
Name: count, dtype: int64


In [73]:
# ============================================================
# FEATURE 3: pickup_day_of_month (1-31)
# ============================================================
# Extracts the day number within the month
# Example: January 5 → 5, January 31 → 31
# WHY: Captures patterns like beginning-of-month (paydays),
#      mid-month, and end-of-month travel differences
# ============================================================

df_m1['pickup_day_of_month'] = df_m1['tpep_pickup_datetime'].dt.day
df_m2['pickup_day_of_month'] = df_m2['tpep_pickup_datetime'].dt.day

print(f"\n✅ Feature 3: pickup_day_of_month")
print(f"   Range: {df_m1['pickup_day_of_month'].min()} to {df_m1['pickup_day_of_month'].max()}")


✅ Feature 3: pickup_day_of_month
   Range: 1 to 31


In [74]:
# ============================================================
# FEATURE 4: is_weekend (0 or 1)
# ============================================================
# Binary flag: 1 if Saturday (5) or Sunday (6), 0 otherwise
# WHY: Weekend travel is fundamentally different from weekday
#      - Weekdays: commuters, business trips, rush hours
#      - Weekends: leisure, tourists, shopping, no rush hour
# ============================================================

df_m1['is_weekend'] = df_m1['pickup_day_of_week'].isin([5, 6]).astype(int)
df_m2['is_weekend'] = df_m2['pickup_day_of_week'].isin([5, 6]).astype(int)

print(f"\n✅ Feature 4: is_weekend")
print(f"   Weekday trips: {(df_m1['is_weekend'] == 0).sum():,}")
print(f"   Weekend trips: {(df_m1['is_weekend'] == 1).sum():,}")
print(f"   Weekend %: {df_m1['is_weekend'].mean() * 100:.1f}%")


✅ Feature 4: is_weekend
   Weekday trips: 2,046,576
   Weekend trips: 674,925
   Weekend %: 24.8%


In [75]:
# ============================================================
# FEATURE 5: is_rush_hour (0 or 1)
# ============================================================
# Binary flag: 1 if WEEKDAY AND (7-9 AM or 5-7 PM)
# IMPORTANT: Rush hour only applies on weekdays!
#   Saturday 8 AM is NOT rush hour — no commuters
# WHY: Rush hour = maximum congestion = longer trip durations
#      and higher chance of being in congestion pricing zone
# ============================================================

df_m1['is_rush_hour'] = (
    (df_m1['pickup_hour'].isin([7, 8, 9, 17, 18, 19])) &
    (df_m1['is_weekend'] == 0)
).astype(int)

df_m2['is_rush_hour'] = (
    (df_m2['pickup_hour'].isin([7, 8, 9, 17, 18, 19])) &
    (df_m2['is_weekend'] == 0)
).astype(int)

print(f"\n✅ Feature 5: is_rush_hour")
print(f"   Rush hour trips: {df_m1['is_rush_hour'].sum():,}")
print(f"   Non-rush hour: {(df_m1['is_rush_hour'] == 0).sum():,}")
print(f"   Rush hour %: {df_m1['is_rush_hour'].mean() * 100:.1f}%")


✅ Feature 5: is_rush_hour
   Rush hour trips: 694,853
   Non-rush hour: 2,026,648
   Rush hour %: 25.5%


In [76]:
# ============================================================
# FEATURE 6: is_peak_hour (0 or 1)
# ============================================================
# Binary flag: 1 if WEEKDAY AND 5-7 PM (evening peak ONLY)
# Different from is_rush_hour — this isolates JUST the evening
# WHY: Evening peak (5-7 PM) is typically the WORST congestion
#      window in NYC. Isolating it gives the model a finer
#      signal than the broader rush_hour flag.
# ============================================================

df_m1['is_peak_hour'] = (
    (df_m1['pickup_hour'].isin([17, 18, 19])) &
    (df_m1['is_weekend'] == 0)
).astype(int)

df_m2['is_peak_hour'] = (
    (df_m2['pickup_hour'].isin([17, 18, 19])) &
    (df_m2['is_weekend'] == 0)
).astype(int)

print(f"\n✅ Feature 6: is_peak_hour")
print(f"   Peak hour trips: {df_m1['is_peak_hour'].sum():,}")
print(f"   Peak hour %: {df_m1['is_peak_hour'].mean() * 100:.1f}%")


✅ Feature 6: is_peak_hour
   Peak hour trips: 447,654
   Peak hour %: 16.4%


In [77]:
# ============================================================
# FEATURE 7: time_period (categorical)
# ============================================================
# Groups the 24 hours into 5 meaningful time blocks:
#   early_morning: 0-5   (midnight to 5 AM — very quiet)
#   morning:       6-11  (6 AM to 11 AM — morning commute)
#   afternoon:     12-16 (noon to 4 PM — midday activity)
#   evening:       17-20 (5 PM to 8 PM — evening rush)
#   night:         21-23 (9 PM to midnight — nightlife)
# WHY: Broader time grouping captures general patterns that
#      the model might miss with just the raw hour number.
#      Example: ALL morning hours (6-11) share commute traits
# NOTE: This will be one-hot encoded later in Phase 4
# ============================================================

bins = [-1, 5, 11, 16, 20, 23]
labels = ['early_morning', 'morning', 'afternoon', 'evening', 'night']

df_m1['time_period'] = pd.cut(df_m1['pickup_hour'], bins=bins, labels=labels)
df_m2['time_period'] = pd.cut(df_m2['pickup_hour'], bins=bins, labels=labels)

print(f"\n✅ Feature 7: time_period")
print(f"   Distribution (Model 1):\n{df_m1['time_period'].value_counts()}")


✅ Feature 7: time_period
   Distribution (Model 1):
time_period
afternoon        849947
evening          727054
morning          586409
night            380879
early_morning    177212
Name: count, dtype: int64


In [78]:
# ============================================================
# FEATURES 8 & 9: pickup_hour_sin / pickup_hour_cos
# ============================================================
# CYCLICAL ENCODING — This is a clever technique!
#
# THE PROBLEM with raw hour numbers:
#   Hour 23 (11 PM) and Hour 0 (midnight) are only 1 hour apart
#   in reality, but the model sees them as 23 units apart!
#   The model thinks 11 PM is closer to noon (12) than midnight (0)
#
# THE SOLUTION — Sine and Cosine transforms:
#   Map hours onto a circle using trigonometry
#   Now 23 and 0 are neighbors on the circle, just like in reality
#
# FORMULA:
#   hour_sin = sin(2π × hour / 24)
#   hour_cos = cos(2π × hour / 24)
#
# WHY TWO columns (sin AND cos)?
#   Using only sin: 6 AM and 6 PM would have the same value!
#   Using BOTH sin and cos: every hour gets a unique (x, y) point
#   on the circle — no ambiguity
#
# VISUAL INTUITION:
#   Imagine a clock face. Each hour is a point on the circle.
#   sin gives the vertical position, cos gives the horizontal.
#   Together they uniquely identify each hour.
# ============================================================

df_m1['pickup_hour_sin'] = np.sin(2 * np.pi * df_m1['pickup_hour'] / 24)
df_m1['pickup_hour_cos'] = np.cos(2 * np.pi * df_m1['pickup_hour'] / 24)

df_m2['pickup_hour_sin'] = np.sin(2 * np.pi * df_m2['pickup_hour'] / 24)
df_m2['pickup_hour_cos'] = np.cos(2 * np.pi * df_m2['pickup_hour'] / 24)

print(f"\n✅ Features 8 & 9: pickup_hour_sin / pickup_hour_cos")
print(f"   Hour 0 (midnight):  sin={np.sin(2*np.pi*0/24):.3f}, cos={np.cos(2*np.pi*0/24):.3f}")
print(f"   Hour 6 (6 AM):      sin={np.sin(2*np.pi*6/24):.3f}, cos={np.cos(2*np.pi*6/24):.3f}")
print(f"   Hour 12 (noon):     sin={np.sin(2*np.pi*12/24):.3f}, cos={np.cos(2*np.pi*12/24):.3f}")
print(f"   Hour 23 (11 PM):    sin={np.sin(2*np.pi*23/24):.3f}, cos={np.cos(2*np.pi*23/24):.3f}")
print(f"   ↑ Notice: Hour 23 and Hour 0 have SIMILAR values (neighbors on circle!)")


✅ Features 8 & 9: pickup_hour_sin / pickup_hour_cos
   Hour 0 (midnight):  sin=0.000, cos=1.000
   Hour 6 (6 AM):      sin=1.000, cos=0.000
   Hour 12 (noon):     sin=0.000, cos=-1.000
   Hour 23 (11 PM):    sin=-0.259, cos=0.966
   ↑ Notice: Hour 23 and Hour 0 have SIMILAR values (neighbors on circle!)


In [79]:
# ============================================================
# FEATURES 10 & 11: pickup_dow_sin / pickup_dow_cos
# ============================================================
# Same cyclical encoding logic, but for day of week
#
# THE PROBLEM: Sunday (6) and Monday (0) are adjacent days,
#   but the model sees them as 6 units apart!
#
# THE SOLUTION: Map days onto a circle using sin/cos
#   Now Sunday and Monday are neighbors, just like in reality
#
# FORMULA:
#   dow_sin = sin(2π × day_of_week / 7)
#   dow_cos = cos(2π × day_of_week / 7)
# ============================================================

df_m1['pickup_dow_sin'] = np.sin(2 * np.pi * df_m1['pickup_day_of_week'] / 7)
df_m1['pickup_dow_cos'] = np.cos(2 * np.pi * df_m1['pickup_day_of_week'] / 7)

df_m2['pickup_dow_sin'] = np.sin(2 * np.pi * df_m2['pickup_day_of_week'] / 7)
df_m2['pickup_dow_cos'] = np.cos(2 * np.pi * df_m2['pickup_day_of_week'] / 7)

print(f"\n✅ Features 10 & 11: pickup_dow_sin / pickup_dow_cos")
print(f"   Monday (0):  sin={np.sin(2*np.pi*0/7):.3f}, cos={np.cos(2*np.pi*0/7):.3f}")
print(f"   Friday (4):  sin={np.sin(2*np.pi*4/7):.3f}, cos={np.cos(2*np.pi*4/7):.3f}")
print(f"   Sunday (6):  sin={np.sin(2*np.pi*6/7):.3f}, cos={np.cos(2*np.pi*6/7):.3f}")
print(f"   ↑ Notice: Sunday (6) and Monday (0) have SIMILAR values!")


✅ Features 10 & 11: pickup_dow_sin / pickup_dow_cos
   Monday (0):  sin=0.000, cos=1.000
   Friday (4):  sin=-0.434, cos=-0.901
   Sunday (6):  sin=-0.782, cos=0.623
   ↑ Notice: Sunday (6) and Monday (0) have SIMILAR values!


In [80]:
# ============================================================
# PHASE 1 SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PHASE 1 COMPLETE: TEMPORAL FEATURES")
print("=" * 70)

temporal_features = [
    'pickup_hour', 'pickup_day_of_week', 'pickup_day_of_month',
    'is_weekend', 'is_rush_hour', 'is_peak_hour', 'time_period',
    'pickup_hour_sin', 'pickup_hour_cos',
    'pickup_dow_sin', 'pickup_dow_cos'
]

print(f"\n✅ {len(temporal_features)} temporal features created:")
for i, f in enumerate(temporal_features, 1):
    print(f"   {i:2d}. {f}")

print(f"\nModel 1 shape: {df_m1.shape[0]:,} rows × {df_m1.shape[1]} columns (was 22)")
print(f"Model 2 shape: {df_m2.shape[0]:,} rows × {df_m2.shape[1]} columns (was 22)")

# Quick verification — check a sample row to confirm features make sense
print(f"\n📋 Sample row verification (Model 1, first row):")
sample = df_m1[['tpep_pickup_datetime'] + temporal_features].iloc[0]
print(sample)


PHASE 1 COMPLETE: TEMPORAL FEATURES

✅ 11 temporal features created:
    1. pickup_hour
    2. pickup_day_of_week
    3. pickup_day_of_month
    4. is_weekend
    5. is_rush_hour
    6. is_peak_hour
    7. time_period
    8. pickup_hour_sin
    9. pickup_hour_cos
   10. pickup_dow_sin
   11. pickup_dow_cos

Model 1 shape: 2,721,501 rows × 33 columns (was 22)
Model 2 shape: 2,412,426 rows × 33 columns (was 22)

📋 Sample row verification (Model 1, first row):
tpep_pickup_datetime    2025-01-01 00:18:38
pickup_hour                               0
pickup_day_of_week                        2
pickup_day_of_month                       1
is_weekend                                0
is_rush_hour                              0
is_peak_hour                              0
time_period                   early_morning
pickup_hour_sin                        0.00
pickup_hour_cos                        1.00
pickup_dow_sin                         0.97
pickup_dow_cos                        -0.22
Name: 0, 

In [81]:
# ============================================================
# ============================================================
#
#   PHASE 2: GEOGRAPHIC & LOCATION FEATURES
#   11 features from pickup/dropoff zone IDs
#
# ============================================================
# ============================================================
# WHY GEOGRAPHIC FEATURES MATTER:
# Think of Maria again — WHERE she picks up a passenger
# tells her even more than WHEN:
# - Times Square pickup → expect heavy traffic, short distance
# - JFK Airport pickup → long trip, flat rate, highway driving
# - Brooklyn residential → moderate traffic, varies by time
#
# For the congestion fee prediction (Model 2), location is
# THE most important factor — the fee applies specifically
# to trips entering/within the Manhattan CBD (south of 60th St).
# ============================================================

In [82]:
# ============================================================
# LOAD TLC TAXI ZONE LOOKUP TABLE
# ============================================================
# This official file maps each LocationID to its Borough,
# Zone name, and service zone. We'll use it to:
# 1. Identify airport zones
# 2. Identify Manhattan zones
# 3. Combine with actual fee data to find CBD zones
#
# SOURCE: NYC TLC Trip Record Data page
# https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
# ============================================================

zone_lookup_path = r"C:\Users\hunda\OneDrive\Desktop\Machine Learning Project\raw\taxi_zone_lookup.csv"
zone_lookup = pd.read_csv(zone_lookup_path)

print("\n" + "=" * 70)
print("PHASE 2: GEOGRAPHIC & LOCATION FEATURES")
print("=" * 70)

print(f"\n📋 TLC Zone Lookup loaded: {zone_lookup.shape[0]} zones")
print(f"   Columns: {list(zone_lookup.columns)}")
print(f"   Boroughs: {zone_lookup['Borough'].unique()}")


PHASE 2: GEOGRAPHIC & LOCATION FEATURES

📋 TLC Zone Lookup loaded: 265 zones
   Columns: ['LocationID', 'Borough', 'Zone', 'service_zone']
   Boroughs: ['EWR' 'Queens' 'Bronx' 'Manhattan' 'Staten Island' 'Brooklyn' 'Unknown'
 nan]


In [83]:
# ============================================================
# VERIFY STARTING POINT
# ============================================================
# Quick sanity check — confirm datasets are clean and match
# what we expect from Module 4
# ============================================================

print("\n" + "=" * 70)
print("STARTING POINT VERIFICATION")
print("=" * 70)

# Model 1 checks
print("\n--- Model 1 (Duration Regression) ---")
print(f"Nulls: {df_m1.isnull().sum().sum()}")
print(f"Duration range: {df_m1['trip_duration_min'].min():.2f} to {df_m1['trip_duration_min'].max():.2f} min")
print(f"Columns: {list(df_m1.columns)}")

# Model 2 checks
print("\n--- Model 2 (Congestion Classification) ---")
print(f"Nulls: {df_m2.isnull().sum().sum()}")
print(f"Target distribution:\n{df_m2['has_congestion_fee'].value_counts()}")
print(f"Earliest pickup: {df_m2['tpep_pickup_datetime'].min()}")


STARTING POINT VERIFICATION

--- Model 1 (Duration Regression) ---
Nulls: 0
Duration range: 1.00 to 179.03 min
Columns: ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee', 'trip_duration_min', 'has_congestion_fee', 'pickup_hour', 'pickup_day_of_week', 'pickup_day_of_month', 'is_weekend', 'is_rush_hour', 'is_peak_hour', 'time_period', 'pickup_hour_sin', 'pickup_hour_cos', 'pickup_dow_sin', 'pickup_dow_cos']

--- Model 2 (Congestion Classification) ---
Nulls: 0
Target distribution:
has_congestion_fee
1    1798485
0     613941
Name: count, dtype: int64
Earliest pickup: 2025-01-05 00:00:00


In [84]:
# ============================================================
# STEP 1: IDENTIFY AIRPORT ZONES (from zone lookup)
# ============================================================
# Instead of hardcoding, we find airports from the data itself.
# Airport zones have service_zone == 'Airports' or 'EWR'
# ============================================================

airport_zones_df = zone_lookup[
    zone_lookup['service_zone'].isin(['Airports', 'EWR'])
]

AIRPORT_ZONES = airport_zones_df['LocationID'].tolist()

print(f"\n--- Airport Zones (from zone lookup) ---")
for _, row in airport_zones_df.iterrows():
    print(f"   Zone {row['LocationID']}: {row['Zone']} ({row['Borough']})")


--- Airport Zones (from zone lookup) ---
   Zone 1: Newark Airport (EWR)
   Zone 132: JFK Airport (Queens)
   Zone 138: LaGuardia Airport (Queens)


In [85]:
# ============================================================
# STEP 2: IDENTIFY CBD ZONES (data-driven approach)
# ============================================================
# THE SMART APPROACH:
# Instead of manually guessing which zones are south of 60th St,
# we let the DATA tell us!
#
# LOGIC:
# 1. Get all Manhattan zones from the TLC lookup
# 2. For each Manhattan zone, calculate what % of trips
#    starting there had the congestion fee charged ($0.75)
# 3. Zones with HIGH fee rates (>50%) are functionally
#    inside the CBD — the fee is almost always charged there
# 4. Zones with LOW fee rates are outside the CBD
#
# WHY THIS WORKS:
# The cbd_congestion_fee column in our data is the GROUND TRUTH.
# It tells us exactly which trips were charged. If 90% of trips
# from Zone 161 (Times Square) have the fee → it's in the CBD.
# If only 5% of trips from Zone 116 (Hamilton Heights) have
# the fee → it's NOT in the CBD.
#
# We use Model 2 data (post-Jan-5 only) because congestion
# pricing started on January 5, 2025.
# ============================================================

# Step 2a: Get all Manhattan zone IDs from lookup
manhattan_zones = zone_lookup[zone_lookup['Borough'] == 'Manhattan']['LocationID'].tolist()
print(f"\n--- Identifying CBD Zones (data-driven) ---")
print(f"   Total Manhattan zones: {len(manhattan_zones)}")

# Step 2b: For each Manhattan zone, calculate fee rate from ACTUAL data
# Fee rate = % of trips from that zone where congestion fee was charged
zone_fee_stats = df_m2[df_m2['PULocationID'].isin(manhattan_zones)].groupby('PULocationID').agg(
    total_trips=('has_congestion_fee', 'count'),
    fee_trips=('has_congestion_fee', 'sum'),
    fee_rate=('has_congestion_fee', 'mean')
).reset_index()

# Step 2c: Zones with fee rate > 50% are functionally in the CBD
# (more than half of trips from there get charged the fee)
CBD_THRESHOLD = 0.50
cbd_zones_df = zone_fee_stats[zone_fee_stats['fee_rate'] > CBD_THRESHOLD]
non_cbd_manhattan_df = zone_fee_stats[zone_fee_stats['fee_rate'] <= CBD_THRESHOLD]

CBD_ZONES = cbd_zones_df['PULocationID'].tolist()

print(f"   CBD zones identified (fee rate > {CBD_THRESHOLD*100:.0f}%): {len(CBD_ZONES)}")
print(f"   Non-CBD Manhattan zones: {len(non_cbd_manhattan_df)}")

# Step 2d: Show the CBD zones with their names and fee rates
cbd_zones_with_names = cbd_zones_df.merge(
    zone_lookup[['LocationID', 'Zone']],
    left_on='PULocationID',
    right_on='LocationID',
    how='left'
).sort_values('fee_rate', ascending=False)

print(f"\n   ✅ CBD Zones (sorted by fee rate):")
print(f"   {'Zone ID':<10} {'Zone Name':<40} {'Fee Rate':<10} {'Trips':<10}")
print(f"   {'-'*70}")
for _, row in cbd_zones_with_names.iterrows():
    print(f"   {int(row['PULocationID']):<10} {row['Zone']:<40} {row['fee_rate']*100:>5.1f}%    {int(row['total_trips']):>8,}")

# Step 2e: Show non-CBD Manhattan zones for comparison
print(f"\n   ❌ Non-CBD Manhattan Zones (fee rate ≤ {CBD_THRESHOLD*100:.0f}%):")
non_cbd_with_names = non_cbd_manhattan_df.merge(
    zone_lookup[['LocationID', 'Zone']],
    left_on='PULocationID',
    right_on='LocationID',
    how='left'
).sort_values('fee_rate', ascending=False)

print(f"   {'Zone ID':<10} {'Zone Name':<40} {'Fee Rate':<10} {'Trips':<10}")
print(f"   {'-'*70}")
for _, row in non_cbd_with_names.iterrows():
    print(f"   {int(row['PULocationID']):<10} {row['Zone']:<40} {row['fee_rate']*100:>5.1f}%    {int(row['total_trips']):>8,}")

# Summary
print(f"\n   📊 Summary:")
print(f"   CBD zones avg fee rate: {cbd_zones_df['fee_rate'].mean()*100:.1f}%")
print(f"   Non-CBD Manhattan avg fee rate: {non_cbd_manhattan_df['fee_rate'].mean()*100:.1f}%")
print(f"   ↑ Large gap confirms the data-driven split is accurate!")



--- Identifying CBD Zones (data-driven) ---
   Total Manhattan zones: 69
   CBD zones identified (fee rate > 50%): 40
   Non-CBD Manhattan zones: 26

   ✅ CBD Zones (sorted by fee rate):
   Zone ID    Zone Name                                Fee Rate   Trips     
   ----------------------------------------------------------------------
   234        Union Sq                                  99.5%      69,086
   170        Murray Hill                               99.5%      69,537
   161        Midtown Center                            99.5%     129,524
   162        Midtown East                              99.5%      93,522
   68         East Chelsea                              99.5%      64,953
   113        Greenwich Village North                   99.5%      34,614
   164        Midtown South                             99.4%      52,776
   163        Midtown North                             99.4%      73,822
   107        Gramercy                                  99.4%      48

In [86]:
# ============================================================
# FEATURE 1: is_same_zone (0 or 1)
# ============================================================
# Binary flag: 1 if pickup and dropoff are in the same zone
# WHY: Same-zone trips are typically very short (a few blocks)
#      and have predictably low duration — strong signal!
# Example: Pickup Zone 161, Dropoff Zone 161 → is_same_zone = 1
# ============================================================

df_m1['is_same_zone'] = (df_m1['PULocationID'] == df_m1['DOLocationID']).astype(int)
df_m2['is_same_zone'] = (df_m2['PULocationID'] == df_m2['DOLocationID']).astype(int)

print(f"\n✅ Feature 1: is_same_zone")
print(f"   Same-zone trips (M1): {df_m1['is_same_zone'].sum():,} ({df_m1['is_same_zone'].mean()*100:.1f}%)")
print(f"   Same-zone trips (M2): {df_m2['is_same_zone'].sum():,} ({df_m2['is_same_zone'].mean()*100:.1f}%)")


✅ Feature 1: is_same_zone
   Same-zone trips (M1): 119,477 (4.4%)
   Same-zone trips (M2): 106,841 (4.4%)


In [87]:
# ============================================================
# FEATURES 2 & 3: pu_is_airport / do_is_airport
# ============================================================
# Binary flags for airport pickups and dropoffs separately
# Airport zones identified from TLC lookup (service_zone)
#
# WHY separate flags?
# - Airport PICKUP: Maria picks up at JFK → long trip to city
# - Airport DROPOFF: Maria drops off at JFK → came from city
# These have different patterns (pickup has flat rate at JFK,
# dropoff was metered from city)
# ============================================================

df_m1['pu_is_airport'] = df_m1['PULocationID'].isin(AIRPORT_ZONES).astype(int)
df_m1['do_is_airport'] = df_m1['DOLocationID'].isin(AIRPORT_ZONES).astype(int)

df_m2['pu_is_airport'] = df_m2['PULocationID'].isin(AIRPORT_ZONES).astype(int)
df_m2['do_is_airport'] = df_m2['DOLocationID'].isin(AIRPORT_ZONES).astype(int)

print(f"\n✅ Features 2 & 3: pu_is_airport / do_is_airport")
print(f"   Airport pickups (M1): {df_m1['pu_is_airport'].sum():,} ({df_m1['pu_is_airport'].mean()*100:.1f}%)")
print(f"   Airport dropoffs (M1): {df_m1['do_is_airport'].sum():,} ({df_m1['do_is_airport'].mean()*100:.1f}%)")


✅ Features 2 & 3: pu_is_airport / do_is_airport
   Airport pickups (M1): 209,823 (7.7%)
   Airport dropoffs (M1): 53,909 (2.0%)


In [88]:
# ============================================================
# FEATURE 4: is_airport_trip (0 or 1)
# ============================================================
# Binary: 1 if EITHER pickup OR dropoff is an airport
# WHY: Simplifies the airport signal into one feature
#      "Is this an airport-related trip at all?"
#      Airport trips tend to be longer, use highways, and
#      have different pricing (JFK has a flat $70 fare)
# ============================================================

df_m1['is_airport_trip'] = (df_m1['pu_is_airport'] | df_m1['do_is_airport']).astype(int)
df_m2['is_airport_trip'] = (df_m2['pu_is_airport'] | df_m2['do_is_airport']).astype(int)

print(f"\n✅ Feature 4: is_airport_trip")
print(f"   Airport trips (M1): {df_m1['is_airport_trip'].sum():,} ({df_m1['is_airport_trip'].mean()*100:.1f}%)")



✅ Feature 4: is_airport_trip
   Airport trips (M1): 258,596 (9.5%)


In [89]:
# ============================================================
# FEATURES 5 & 6: pu_is_cbd / do_is_cbd
# ============================================================
# Binary flags: 1 if zone is within Manhattan CBD
# (south of 60th Street — the Congestion Relief Zone)
#
# CBD_ZONES was identified using the DATA-DRIVEN approach above
# (zones where >50% of trips have the congestion fee charged)
#
# WHY THIS IS CRITICAL:
# For Model 2 (congestion fee prediction), this is the
# MOST IMPORTANT feature! The congestion fee is literally
# defined by whether the trip goes to/from/within the CBD.
#
# Think of it as: "Did Maria's taxi enter the toll zone?"
# If pickup is in CBD → she's already in the zone
# If dropoff is in CBD → she's driving INTO the zone
# ============================================================

df_m1['pu_is_cbd'] = df_m1['PULocationID'].isin(CBD_ZONES).astype(int)
df_m1['do_is_cbd'] = df_m1['DOLocationID'].isin(CBD_ZONES).astype(int)

df_m2['pu_is_cbd'] = df_m2['PULocationID'].isin(CBD_ZONES).astype(int)
df_m2['do_is_cbd'] = df_m2['DOLocationID'].isin(CBD_ZONES).astype(int)

print(f"\n✅ Features 5 & 6: pu_is_cbd / do_is_cbd")
print(f"   CBD pickups (M1): {df_m1['pu_is_cbd'].sum():,} ({df_m1['pu_is_cbd'].mean()*100:.1f}%)")
print(f"   CBD dropoffs (M1): {df_m1['do_is_cbd'].sum():,} ({df_m1['do_is_cbd'].mean()*100:.1f}%)")
print(f"   CBD pickups (M2): {df_m2['pu_is_cbd'].sum():,} ({df_m2['pu_is_cbd'].mean()*100:.1f}%)")
print(f"   CBD dropoffs (M2): {df_m2['do_is_cbd'].sum():,} ({df_m2['do_is_cbd'].mean()*100:.1f}%)")


✅ Features 5 & 6: pu_is_cbd / do_is_cbd
   CBD pickups (M1): 1,739,932 (63.9%)
   CBD dropoffs (M1): 1,633,529 (60.0%)
   CBD pickups (M2): 1,538,446 (63.8%)
   CBD dropoffs (M2): 1,444,144 (59.9%)


In [90]:
# ============================================================
# FEATURE 7: cbd_trip_type (categorical)
# ============================================================
# Categorizes each trip based on its CBD relationship:
#   'within_cbd'  → Both pickup AND dropoff in CBD
#   'into_cbd'    → Pickup outside, dropoff inside CBD
#   'out_of_cbd'  → Pickup inside, dropoff outside CBD
#   'outside_cbd' → Neither pickup nor dropoff in CBD
#
# WHY: Each type has very different characteristics:
#   - within_cbd: Short trips, always has congestion fee
#   - into_cbd: Heading into Manhattan, likely has fee
#   - out_of_cbd: Leaving Manhattan, may or may not have fee
#   - outside_cbd: Brooklyn to Queens etc., typically no fee
#
# IMPLEMENTATION NOTE:
# We use np.select() instead of .apply() for SPEED.
# .apply() processes row-by-row (slow on 2.7M rows)
# np.select() processes entire columns at once (fast!)
#
# NOTE: Will be one-hot encoded in Phase 4
# ============================================================

# Define conditions using vectorized operations (FAST)
conditions_m1 = [
    (df_m1['pu_is_cbd'] == 1) & (df_m1['do_is_cbd'] == 1),  # both in CBD
    (df_m1['pu_is_cbd'] == 0) & (df_m1['do_is_cbd'] == 1),  # going into CBD
    (df_m1['pu_is_cbd'] == 1) & (df_m1['do_is_cbd'] == 0),  # leaving CBD
    (df_m1['pu_is_cbd'] == 0) & (df_m1['do_is_cbd'] == 0),  # outside CBD
]
choices = ['within_cbd', 'into_cbd', 'out_of_cbd', 'outside_cbd']

df_m1['cbd_trip_type'] = np.select(conditions_m1, choices, default='outside_cbd')

conditions_m2 = [
    (df_m2['pu_is_cbd'] == 1) & (df_m2['do_is_cbd'] == 1),
    (df_m2['pu_is_cbd'] == 0) & (df_m2['do_is_cbd'] == 1),
    (df_m2['pu_is_cbd'] == 1) & (df_m2['do_is_cbd'] == 0),
    (df_m2['pu_is_cbd'] == 0) & (df_m2['do_is_cbd'] == 0),
]

df_m2['cbd_trip_type'] = np.select(conditions_m2, choices, default='outside_cbd')

print(f"\n✅ Feature 7: cbd_trip_type")
print(f"   Distribution (Model 1):\n{pd.Series(df_m1['cbd_trip_type']).value_counts()}")
print(f"\n   Distribution (Model 2):\n{pd.Series(df_m2['cbd_trip_type']).value_counts()}")


✅ Feature 7: cbd_trip_type
   Distribution (Model 1):
cbd_trip_type
within_cbd     1262208
outside_cbd     610248
out_of_cbd      477724
into_cbd        371321
Name: count, dtype: int64

   Distribution (Model 2):
cbd_trip_type
within_cbd     1113724
outside_cbd     543560
out_of_cbd      424722
into_cbd        330420
Name: count, dtype: int64


In [91]:
# ============================================================
# FEATURES 8 & 9: pickup_zone_popularity / dropoff_zone_popularity
# ============================================================
# Trip count per zone — how many trips originate from or
# arrive at each zone. Measures how "busy" a zone is.
#
# WHY: A zone with 50,000 trips (like Times Square) behaves
#      very differently from a zone with 500 trips (like a
#      quiet residential area). This is a proxy for:
#      - Traffic density
#      - Taxi demand
#      - Urban activity level
#
# IMPORTANT: For now, we compute on the full dataset.
# During train/test split, we'll recompute on TRAINING
# data only and map to test — to avoid data leakage.
# ============================================================

# Pickup zone popularity (trip counts per pickup zone)
pu_popularity_m1 = df_m1['PULocationID'].value_counts().to_dict()
pu_popularity_m2 = df_m2['PULocationID'].value_counts().to_dict()

df_m1['pickup_zone_popularity'] = df_m1['PULocationID'].map(pu_popularity_m1)
df_m2['pickup_zone_popularity'] = df_m2['PULocationID'].map(pu_popularity_m2)

# Dropoff zone popularity (trip counts per dropoff zone)
do_popularity_m1 = df_m1['DOLocationID'].value_counts().to_dict()
do_popularity_m2 = df_m2['DOLocationID'].value_counts().to_dict()

df_m1['dropoff_zone_popularity'] = df_m1['DOLocationID'].map(do_popularity_m1)
df_m2['dropoff_zone_popularity'] = df_m2['DOLocationID'].map(do_popularity_m2)

print(f"\n✅ Features 8 & 9: pickup_zone_popularity / dropoff_zone_popularity")
print(f"   Pickup popularity range (M1): {df_m1['pickup_zone_popularity'].min():,} to {df_m1['pickup_zone_popularity'].max():,}")
print(f"   Dropoff popularity range (M1): {df_m1['dropoff_zone_popularity'].min():,} to {df_m1['dropoff_zone_popularity'].max():,}")
print(f"   Top 5 pickup zones (M1):")
top5_pu = df_m1['PULocationID'].value_counts().head()
for zone_id, count in top5_pu.items():
    zone_name = zone_lookup[zone_lookup['LocationID'] == zone_id]['Zone'].values[0]
    print(f"      Zone {zone_id} ({zone_name}): {count:,} trips")


✅ Features 8 & 9: pickup_zone_popularity / dropoff_zone_popularity
   Pickup popularity range (M1): 1 to 145,956
   Dropoff popularity range (M1): 1 to 142,560
   Top 5 pickup zones (M1):
      Zone 237 (Upper East Side South): 145,956 trips
      Zone 161 (Midtown Center): 144,127 trips
      Zone 236 (Upper East Side North): 136,128 trips
      Zone 132 (JFK Airport): 126,348 trips
      Zone 186 (Penn Station/Madison Sq West): 105,677 trips


In [92]:
# ============================================================
# FEATURES 10 & 11: pu_zone_cluster / do_zone_cluster
# ============================================================
# Categorical: 'high', 'medium', 'low' traffic zones
# Binned from the popularity scores using quantile cuts (terciles)
#
# WHY: Tree-based models (Random Forest, XGBoost) can benefit
#      from binned versions of continuous features. This captures
#      the zone "tier" as a simpler, more generalizable signal.
#
# Tercile split:
#   'low'    → bottom 1/3 of zones by trip count
#   'medium' → middle 1/3
#   'high'   → top 1/3
#
# NOTE: Will be one-hot encoded in Phase 4
# ============================================================

# Create popularity per zone (not per row) for binning
pu_zone_counts_m1 = df_m1['PULocationID'].value_counts()
do_zone_counts_m1 = df_m1['DOLocationID'].value_counts()

pu_zone_counts_m2 = df_m2['PULocationID'].value_counts()
do_zone_counts_m2 = df_m2['DOLocationID'].value_counts()

# Bin zones into 3 tiers using quantile cuts
pu_zone_tier_m1 = pd.qcut(pu_zone_counts_m1, q=3, labels=['low', 'medium', 'high']).to_dict()
do_zone_tier_m1 = pd.qcut(do_zone_counts_m1, q=3, labels=['low', 'medium', 'high']).to_dict()

pu_zone_tier_m2 = pd.qcut(pu_zone_counts_m2, q=3, labels=['low', 'medium', 'high']).to_dict()
do_zone_tier_m2 = pd.qcut(do_zone_counts_m2, q=3, labels=['low', 'medium', 'high']).to_dict()

# Map back to rows
df_m1['pu_zone_cluster'] = df_m1['PULocationID'].map(pu_zone_tier_m1)
df_m1['do_zone_cluster'] = df_m1['DOLocationID'].map(do_zone_tier_m1)

df_m2['pu_zone_cluster'] = df_m2['PULocationID'].map(pu_zone_tier_m2)
df_m2['do_zone_cluster'] = df_m2['DOLocationID'].map(do_zone_tier_m2)

print(f"\n✅ Features 10 & 11: pu_zone_cluster / do_zone_cluster")
print(f"   Pickup zone cluster distribution (M1):\n{df_m1['pu_zone_cluster'].value_counts()}")
print(f"\n   Dropoff zone cluster distribution (M1):\n{df_m1['do_zone_cluster'].value_counts()}")


✅ Features 10 & 11: pu_zone_cluster / do_zone_cluster
   Pickup zone cluster distribution (M1):
pu_zone_cluster
high      2714885
medium       6007
low           609
Name: count, dtype: int64

   Dropoff zone cluster distribution (M1):
do_zone_cluster
high      2648416
medium      63005
low         10080
Name: count, dtype: int64


In [93]:
# ============================================================
# PHASE 2 SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PHASE 2 COMPLETE: GEOGRAPHIC & LOCATION FEATURES")
print("=" * 70)

geographic_features = [
    'is_same_zone',
    'pu_is_airport', 'do_is_airport', 'is_airport_trip',
    'pu_is_cbd', 'do_is_cbd', 'cbd_trip_type',
    'pickup_zone_popularity', 'dropoff_zone_popularity',
    'pu_zone_cluster', 'do_zone_cluster'
]

print(f"\n✅ {len(geographic_features)} geographic features created:")
for i, f in enumerate(geographic_features, 1):
    print(f"   {i:2d}. {f}")

print(f"\nModel 1 shape: {df_m1.shape[0]:,} rows × {df_m1.shape[1]} columns (was 33 after Phase 1)")
print(f"Model 2 shape: {df_m2.shape[0]:,} rows × {df_m2.shape[1]} columns (was 33 after Phase 1)")

# CRITICAL VALIDATION: CBD trip type vs actual congestion fee
# This is the ultimate test — does our geographic classification
# actually align with when the fee is charged?
print(f"\n📋 VALIDATION: CBD trip type vs actual congestion fee (Model 2):")
validation = df_m2.groupby('cbd_trip_type')['has_congestion_fee'].agg(['mean', 'count'])
validation.columns = ['fee_rate', 'trip_count']
validation['fee_rate'] = (validation['fee_rate'] * 100).round(1)
print(validation.to_string())
print(f"\n   ↑ within_cbd & into_cbd should have HIGH fee rates")
print(f"   ↑ outside_cbd should have LOW fee rate")
print(f"   ↑ If this pattern holds, our CBD zones are accurate!")


PHASE 2 COMPLETE: GEOGRAPHIC & LOCATION FEATURES

✅ 11 geographic features created:
    1. is_same_zone
    2. pu_is_airport
    3. do_is_airport
    4. is_airport_trip
    5. pu_is_cbd
    6. do_is_cbd
    7. cbd_trip_type
    8. pickup_zone_popularity
    9. dropoff_zone_popularity
   10. pu_zone_cluster
   11. do_zone_cluster

Model 1 shape: 2,721,501 rows × 44 columns (was 33 after Phase 1)
Model 2 shape: 2,412,426 rows × 44 columns (was 33 after Phase 1)

📋 VALIDATION: CBD trip type vs actual congestion fee (Model 2):
               fee_rate  trip_count
cbd_trip_type                      
into_cbd          84.80      330420
out_of_cbd        88.50      424722
outside_cbd        7.80      543560
within_cbd        98.80     1113724

   ↑ within_cbd & into_cbd should have HIGH fee rates
   ↑ outside_cbd should have LOW fee rate
   ↑ If this pattern holds, our CBD zones are accurate!


In [94]:
# ============================================================
# ============================================================
#
#   PHASE 3: TRIP CHARACTERISTICS & DERIVED METRICS
#   10 features from trip distance, fare, and passenger data
#
# ============================================================
# ============================================================
# WHY TRIP CHARACTERISTICS MATTER:
# Phase 1 told us WHEN the trip happens (time features)
# Phase 2 told us WHERE the trip happens (location features)
# Phase 3 tells us WHAT the trip looks like:
# - How far? (distance features)
# - How many passengers? (passenger features)
# - How expensive? (fare features — for analysis only)
# - Is this trip unusual? (outlier flags)
#
# IMPORTANT — DATA LEAKAGE REMINDER:
# Some features here use information only available AFTER the
# trip ends (fare, tip, duration). These are created for
# STAKEHOLDER ANALYSIS (Maria's profitability insights) but
# CANNOT be used in prediction models.
#
# Think of Maria at the moment of pickup:
# ✅ She KNOWS: distance (passenger told her destination),
#    passenger count (she can see them)
# ❌ She DOESN'T KNOW: fare, tip, speed, cost per minute
#    (trip hasn't happened yet!)
# ============================================================


print("\n" + "=" * 70)
print("PHASE 3: TRIP CHARACTERISTICS & DERIVED METRICS")
print("=" * 70)


PHASE 3: TRIP CHARACTERISTICS & DERIVED METRICS


In [95]:
# ============================================================
# FEATURE 1: log_trip_distance
# ============================================================
# Natural log transform of trip_distance
# Uses np.log1p (log of 1 + x) which safely handles small values
#
# WHY LOG TRANSFORM?
# Trip distance is heavily right-skewed:
#   - MANY trips are 1-3 miles (short Manhattan rides)
#   - FEW trips are 20-50+ miles (airport runs)
#
# This skew makes it hard for models to learn because the
# few extreme values dominate. Log transform "squishes" the
# right tail and makes the distribution more bell-shaped.
#
# ANALOGY: Imagine a class photo where one student is 7 feet
# tall. The photographer would have trouble framing everyone.
# Log transform is like asking the tall student to bend
# their knees a bit — now everyone fits in the frame and
# you can see all faces clearly.
#
# USED IN: Both models (distance is known before the trip)
# ============================================================

df_m1['log_trip_distance'] = np.log1p(df_m1['trip_distance'])
df_m2['log_trip_distance'] = np.log1p(df_m2['trip_distance'])

print(f"\n✅ Feature 1: log_trip_distance")
print(f"   Original distance range: {df_m1['trip_distance'].min():.2f} to {df_m1['trip_distance'].max():.2f} miles")
print(f"   Log distance range: {df_m1['log_trip_distance'].min():.2f} to {df_m1['log_trip_distance'].max():.2f}")
print(f"   Original skewness: {df_m1['trip_distance'].skew():.2f}")
print(f"   Log skewness: {df_m1['log_trip_distance'].skew():.2f}")
print(f"   ↑ Closer to 0 = more symmetric = better for models!")


✅ Feature 1: log_trip_distance
   Original distance range: 0.01 to 97.58 miles
   Log distance range: 0.01 to 4.59
   Original skewness: 3.22
   Log skewness: 1.32
   ↑ Closer to 0 = more symmetric = better for models!


In [96]:
# ============================================================
# FEATURE 2: distance_category
# ============================================================
# Bins trip distance into 4 meaningful categories:
#   'short'     → 0-2 miles   (quick Manhattan ride)
#   'medium'    → 2-5 miles   (cross-town or short borough)
#   'long'      → 5-10 miles  (borough-to-borough)
#   'very_long' → 10+ miles   (airport runs, outer borough)
#
# WHY BINS?
# The relationship between distance and duration isn't linear:
# - Going from 1 to 2 miles might add 5 minutes (city traffic)
# - Going from 30 to 31 miles might add only 1 minute (highway)
#
# Bins capture these non-linear jumps. The model can learn:
# "short trips behave THIS way, long trips behave THAT way"
#
# USED IN: Both models (distance is known before the trip)
# NOTE: Will be one-hot encoded in Phase 4
# ============================================================

bins = [0, 2, 5, 10, 100]
labels = ['short', 'medium', 'long', 'very_long']

df_m1['distance_category'] = pd.cut(df_m1['trip_distance'], bins=bins, labels=labels)
df_m2['distance_category'] = pd.cut(df_m2['trip_distance'], bins=bins, labels=labels)

print(f"\n✅ Feature 2: distance_category")
print(f"   Distribution (Model 1):\n{df_m1['distance_category'].value_counts()}")


✅ Feature 2: distance_category
   Distribution (Model 1):
distance_category
short        1654377
medium        675379
long          199693
very_long     192052
Name: count, dtype: int64


In [97]:
# ============================================================
# FEATURE 3: is_single_passenger (0 or 1)
# ============================================================
# Binary flag: 1 if passenger_count == 1, 0 otherwise
#
# WHY?
# Solo riders make up the VAST majority of taxi trips.
# They may have different patterns than groups:
# - Solo: commuters, business travelers, quick errands
# - Groups: tourists, airport trips, night out with friends
#
# USED IN: Both models (passenger count is known at pickup)
# ============================================================

df_m1['is_single_passenger'] = (df_m1['passenger_count'] == 1).astype(int)
df_m2['is_single_passenger'] = (df_m2['passenger_count'] == 1).astype(int)

print(f"\n✅ Feature 3: is_single_passenger")
print(f"   Solo riders (M1): {df_m1['is_single_passenger'].sum():,} ({df_m1['is_single_passenger'].mean()*100:.1f}%)")
print(f"   Group riders (M1): {(df_m1['is_single_passenger'] == 0).sum():,} ({(df_m1['is_single_passenger'] == 0).mean()*100:.1f}%)")


✅ Feature 3: is_single_passenger
   Solo riders (M1): 2,168,457 (79.7%)
   Group riders (M1): 553,044 (20.3%)


In [98]:
# ============================================================
# FEATURE 4: passenger_count_binned
# ============================================================
# Groups passengers into meaningful categories:
#   '1'  → Solo rider
#   '2'  → Pair (couple, friends)
#   '3+' → Group (family, tourists)
#
# WHY BIN INSTEAD OF RAW COUNT?
# The difference between 1 and 2 passengers is meaningful
# (solo commuter vs couple). But the difference between
# 4 and 5 passengers? Not so much — they're both "a group."
#
# USED IN: Both models (passenger count is known at pickup)
# NOTE: Will be one-hot encoded in Phase 4 
# ============================================================

df_m1['passenger_count_binned'] = pd.cut(
    df_m1['passenger_count'],
    bins=[0, 1, 2, 6],
    labels=['1', '2', '3+']
)

df_m2['passenger_count_binned'] = pd.cut(
    df_m2['passenger_count'],
    bins=[0, 1, 2, 6],
    labels=['1', '2', '3+']
)

print(f"\n✅ Feature 4: passenger_count_binned")
print(f"   Distribution (Model 1):\n{df_m1['passenger_count_binned'].value_counts()}")


✅ Feature 4: passenger_count_binned
   Distribution (Model 1):
passenger_count_binned
1     2168457
2      385265
3+     167779
Name: count, dtype: int64


In [99]:
# ============================================================
# FEATURE 5: is_extreme_distance (0 or 1)
# ============================================================
# Binary flag: 1 if trip_distance is in the top 1% of all trips
#
# WHY?
# Even after cleaning (we removed trips > 100 miles), trips
# near the boundary (say 40-99 miles) are still unusual.
# These edge-case trips behave differently from typical trips:
# - They're almost always highway/airport trips
# - Duration prediction is harder (highway vs city driving)
# - Fare structure may differ (negotiated rates)
#
# By flagging them, the model can learn:
# "When this flag is ON, use different rules"
#
# USED IN: Both models (distance is known before the trip)
# ============================================================

distance_99th = df_m1['trip_distance'].quantile(0.99)

df_m1['is_extreme_distance'] = (df_m1['trip_distance'] > distance_99th).astype(int)
df_m2['is_extreme_distance'] = (df_m2['trip_distance'] > distance_99th).astype(int)

print(f"\n✅ Feature 5: is_extreme_distance")
print(f"   99th percentile threshold: {distance_99th:.2f} miles")
print(f"   Extreme distance trips (M1): {df_m1['is_extreme_distance'].sum():,} ({df_m1['is_extreme_distance'].mean()*100:.1f}%)")


✅ Feature 5: is_extreme_distance
   99th percentile threshold: 19.70 miles
   Extreme distance trips (M1): 27,159 (1.0%)


In [100]:
# ============================================================
# FEATURE 6: is_extreme_fare (0 or 1)
# ============================================================
# Binary flag: 1 if fare_amount is in the top 1% of all trips
#
# WHY?
# Extremely high fares indicate unusual trips:
# - Long-distance runs
# - Special rate codes (negotiated fares)
# - Airport flat rates
# These trips have different characteristics worth flagging.
#
# USED IN: Model 1 only (fare is known after trip, but this
#   flag correlates with distance which IS known before trip)
# For Model 2: Analysis only (fare isn't known before trip)
# ============================================================

fare_99th = df_m1['fare_amount'].quantile(0.99)

df_m1['is_extreme_fare'] = (df_m1['fare_amount'] > fare_99th).astype(int)
df_m2['is_extreme_fare'] = (df_m2['fare_amount'] > fare_99th).astype(int)

print(f"\n✅ Feature 6: is_extreme_fare")
print(f"   99th percentile threshold: ${fare_99th:.2f}")
print(f"   Extreme fare trips (M1): {df_m1['is_extreme_fare'].sum():,} ({df_m1['is_extreme_fare'].mean()*100:.1f}%)")


✅ Feature 6: is_extreme_fare
   99th percentile threshold: $71.60
   Extreme fare trips (M1): 27,179 (1.0%)


In [101]:
# ============================================================
# ============================================================
#   ANALYSIS-ONLY FEATURES (Features 7-10)
# ============================================================
# The next 4 features use POST-TRIP information (fare, tip,
# duration). They CANNOT be used for prediction because
# they aren't known at the moment Maria accepts the ride.
#
# BUT they are valuable for:
# - Stakeholder analysis (Maria's profitability insights)
# - Understanding trip economics
# - Project report visualizations
#
# We create them, then later in Phase 5 we'll separate them
# into a dedicated analysis dataframe before modeling.
# ============================================================

print(f"\n{'─'*70}")
print(f"   ANALYSIS-ONLY FEATURES (cannot be used in models)")
print(f"{'─'*70}")


──────────────────────────────────────────────────────────────────────
   ANALYSIS-ONLY FEATURES (cannot be used in models)
──────────────────────────────────────────────────────────────────────


In [102]:
# ============================================================
# FEATURE 7: avg_speed_mph (ANALYSIS ONLY ⚠️)
# ============================================================
# Average speed = distance / time
# Formula: trip_distance / (trip_duration_min / 60)
#   → miles / hours = miles per hour
#
# WHY ANALYSIS ONLY?
# Speed requires BOTH distance AND duration.
# Duration is what we're trying to PREDICT in Model 1!
# Using it as a feature would be like peeking at the answer.
#
# BUT it's great for analysis:
# - What's the average speed in rush hour vs late night?
# - Which zones have the slowest traffic?
# - How does speed vary by day of week?
# ============================================================

df_m1['avg_speed_mph'] = df_m1['trip_distance'] / (df_m1['trip_duration_min'] / 60)
df_m2['avg_speed_mph'] = df_m2['trip_distance'] / (df_m2['trip_duration_min'] / 60)

print(f"\n✅ Feature 7: avg_speed_mph ⚠️ ANALYSIS ONLY")
print(f"   Mean speed (M1): {df_m1['avg_speed_mph'].mean():.1f} mph")
print(f"   Median speed (M1): {df_m1['avg_speed_mph'].median():.1f} mph")
print(f"   Min speed (M1): {df_m1['avg_speed_mph'].min():.2f} mph")
print(f"   Max speed (M1): {df_m1['avg_speed_mph'].max():.1f} mph")


✅ Feature 7: avg_speed_mph ⚠️ ANALYSIS ONLY
   Mean speed (M1): 11.3 mph
   Median speed (M1): 9.5 mph
   Min speed (M1): 0.01 mph
   Max speed (M1): 882.3 mph


In [103]:
# ============================================================
# FEATURE 8: fare_per_mile (ANALYSIS ONLY ⚠️)
# ============================================================
# Fare efficiency = fare_amount / trip_distance
#   → dollars per mile
#
# WHY ANALYSIS ONLY?
# Fare is determined AFTER the trip (meter runs during trip).
# Maria doesn't know the fare until the trip ends.
#
# BUT it's great for stakeholder analysis:
# - Which zones give Maria the best $/mile?
# - Are short trips more profitable per mile than long ones?
# - How does fare efficiency vary by time of day?
# ============================================================

df_m1['fare_per_mile'] = df_m1['fare_amount'] / df_m1['trip_distance']
df_m2['fare_per_mile'] = df_m2['fare_amount'] / df_m2['trip_distance']

print(f"\n✅ Feature 8: fare_per_mile ⚠️ ANALYSIS ONLY")
print(f"   Mean fare/mile (M1): ${df_m1['fare_per_mile'].mean():.2f}")
print(f"   Median fare/mile (M1): ${df_m1['fare_per_mile'].median():.2f}")


✅ Feature 8: fare_per_mile ⚠️ ANALYSIS ONLY
   Mean fare/mile (M1): $8.08
   Median fare/mile (M1): $7.26


In [104]:
# ============================================================
# FEATURE 9: tip_percentage (ANALYSIS ONLY ⚠️)
# ============================================================
# Tip as percentage of fare = (tip_amount / fare_amount) × 100
#
# WHY ANALYSIS ONLY?
# Tip is given AFTER the trip, not before.
#
# USEFUL FOR:
# - Tipping behavior patterns (CBD vs outer boroughs)
# - Payment method correlation (credit card tips vs cash)
# - Time-of-day tipping patterns
# ============================================================

df_m1['tip_percentage'] = (df_m1['tip_amount'] / df_m1['fare_amount']) * 100
df_m2['tip_percentage'] = (df_m2['tip_amount'] / df_m2['fare_amount']) * 100

print(f"\n✅ Feature 9: tip_percentage ⚠️ ANALYSIS ONLY")
print(f"   Mean tip % (M1): {df_m1['tip_percentage'].mean():.1f}%")
print(f"   Median tip % (M1): {df_m1['tip_percentage'].median():.1f}%")


✅ Feature 9: tip_percentage ⚠️ ANALYSIS ONLY
   Mean tip % (M1): 23.0%
   Median tip % (M1): 25.9%


In [105]:
# ============================================================
# FEATURE 10: cost_per_minute (ANALYSIS ONLY ⚠️)
# ============================================================
# Revenue rate = total_amount / trip_duration_min
#   → dollars per minute of driving
#
# WHY ANALYSIS ONLY?
# Both total_amount and duration are post-trip values.
#
# THIS IS THE KEY STAKEHOLDER METRIC:
# Maria's core question: "How much do I earn per minute?"
# - Short CBD trip: $16 / 10 min = $1.60/min → good!
# - Long airport trip: $65 / 55 min = $1.18/min → less good
# This helps Maria decide which trips to prioritize.
# ============================================================

df_m1['cost_per_minute'] = df_m1['total_amount'] / df_m1['trip_duration_min']
df_m2['cost_per_minute'] = df_m2['total_amount'] / df_m2['trip_duration_min']

print(f"\n✅ Feature 10: cost_per_minute ⚠️ ANALYSIS ONLY")
print(f"   Mean $/minute (M1): ${df_m1['cost_per_minute'].mean():.2f}")
print(f"   Median $/minute (M1): ${df_m1['cost_per_minute'].median():.2f}")


✅ Feature 10: cost_per_minute ⚠️ ANALYSIS ONLY
   Mean $/minute (M1): $2.18
   Median $/minute (M1): $1.95


In [106]:
# ============================================================
# CHECK FOR INFINITE VALUES
# ============================================================
# Division features can produce infinity (e.g., if trip_distance
# is extremely small, fare_per_mile becomes huge).
# We check and handle these before proceeding.
# ============================================================

print(f"\n--- Checking for infinite values ---")
inf_cols = ['avg_speed_mph', 'fare_per_mile', 'tip_percentage', 'cost_per_minute']

for col in inf_cols:
    inf_count_m1 = np.isinf(df_m1[col]).sum()
    inf_count_m2 = np.isinf(df_m2[col]).sum()
    if inf_count_m1 > 0 or inf_count_m2 > 0:
        print(f"   ⚠️ {col}: {inf_count_m1:,} inf in M1, {inf_count_m2:,} inf in M2 → replacing with NaN")
        df_m1[col] = df_m1[col].replace([np.inf, -np.inf], np.nan)
        df_m2[col] = df_m2[col].replace([np.inf, -np.inf], np.nan)
    else:
        print(f"   ✅ {col}: No infinite values")

# Check for NaN introduced
nan_m1 = df_m1[inf_cols].isnull().sum()
nan_m2 = df_m2[inf_cols].isnull().sum()
if nan_m1.sum() > 0 or nan_m2.sum() > 0:
    print(f"\n   NaN values after inf replacement:")
    print(f"   Model 1:\n{nan_m1[nan_m1 > 0]}")
    print(f"   Model 2:\n{nan_m2[nan_m2 > 0]}")
    print(f"   (These are in analysis-only features — won't affect models)")
else:
    print(f"   ✅ No NaN values introduced")


--- Checking for infinite values ---
   ✅ avg_speed_mph: No infinite values
   ✅ fare_per_mile: No infinite values
   ✅ tip_percentage: No infinite values
   ✅ cost_per_minute: No infinite values
   ✅ No NaN values introduced


In [107]:
# ============================================================
# PHASE 3 SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PHASE 3 COMPLETE: TRIP CHARACTERISTICS & DERIVED METRICS")
print("=" * 70)

modeling_features = [
    'log_trip_distance', 'distance_category',
    'is_single_passenger', 'passenger_count_binned',
    'is_extreme_distance', 'is_extreme_fare'
]

analysis_features = [
    'avg_speed_mph', 'fare_per_mile',
    'tip_percentage', 'cost_per_minute'
]

print(f"\n✅ 6 MODELING features (can be used in models):")
for i, f in enumerate(modeling_features, 1):
    print(f"   {i}. {f}")

print(f"\n⚠️ 4 ANALYSIS-ONLY features (stakeholder insights only):")
for i, f in enumerate(analysis_features, 1):
    print(f"   {i}. {f}")

print(f"\nModel 1 shape: {df_m1.shape[0]:,} rows × {df_m1.shape[1]} columns (was 44 after Phase 2)")
print(f"Model 2 shape: {df_m2.shape[0]:,} rows × {df_m2.shape[1]} columns (was 44 after Phase 2)")


PHASE 3 COMPLETE: TRIP CHARACTERISTICS & DERIVED METRICS

✅ 6 MODELING features (can be used in models):
   1. log_trip_distance
   2. distance_category
   3. is_single_passenger
   4. passenger_count_binned
   5. is_extreme_distance
   6. is_extreme_fare

⚠️ 4 ANALYSIS-ONLY features (stakeholder insights only):
   1. avg_speed_mph
   2. fare_per_mile
   3. tip_percentage
   4. cost_per_minute

Model 1 shape: 2,721,501 rows × 54 columns (was 44 after Phase 2)
Model 2 shape: 2,412,426 rows × 54 columns (was 44 after Phase 2)


In [108]:
# ============================================================
# ============================================================
#
#   PHASE 4: CATEGORICAL ENCODING & INTERACTION FEATURES
#   6 features (expanding to ~16+ columns after encoding)
#
# ============================================================
# ============================================================
# WHY ENCODING MATTERS:
# ML models work with NUMBERS, not text or categories.
# If you feed "credit card" or "morning" into a model,
# it won't know what to do with it.
#
# ONE-HOT ENCODING converts categories into binary columns:
#   payment_type = "credit card" becomes → pay_1 = 1, pay_2 = 0
#   payment_type = "cash"        becomes → pay_1 = 0, pay_2 = 1
#
# Think of it like a form with checkboxes:
#   ☑ Credit card  ☐ Cash  ☐ Flex Fare
# Each checkbox becomes its own column (1 = checked, 0 = not)
#
# WHY INTERACTION FEATURES MATTER:
# Sometimes two features COMBINED tell a different story
# than each one alone. Example:
# - "8 AM" alone → could be busy or quiet
# - "Monday" alone → could be busy or quiet
# - "8 AM on Monday" → DEFINITELY rush hour!
# Interaction features capture these combined effects.
# ============================================================


print("\n" + "=" * 70)
print("PHASE 4: CATEGORICAL ENCODING & INTERACTION FEATURES")
print("=" * 70)


PHASE 4: CATEGORICAL ENCODING & INTERACTION FEATURES


In [109]:
# ============================================================
# ENCODING 1: payment_type → One-Hot
# ============================================================
# Original values in your data:
#   0 = Flex Fare trip
#   1 = Credit card
#   2 = Cash
#
# After encoding:
#   pay_0 = 1 if Flex Fare, 0 otherwise
#   pay_1 = 1 if Credit card, 0 otherwise
#   pay_2 = 1 if Cash, 0 otherwise
#
# WHY: Although payment_type is a number (0, 1, 2), the model
# would treat it as continuous — thinking 2 (Cash) is "bigger"
# than 1 (Credit card), which makes no sense. One-hot encoding
# treats each payment method as independent.
# ============================================================

print(f"\n--- Before encoding ---")
print(f"   payment_type values (M1): {sorted(df_m1['payment_type'].unique())}")

df_m1 = pd.get_dummies(df_m1, columns=['payment_type'], prefix='pay', dtype=int)
df_m2 = pd.get_dummies(df_m2, columns=['payment_type'], prefix='pay', dtype=int)

pay_cols = [c for c in df_m1.columns if c.startswith('pay_')]
print(f"\n✅ Encoding 1: payment_type → {len(pay_cols)} columns: {pay_cols}")
print(f"   Distribution (M1):")
for col in pay_cols:
    print(f"      {col}: {df_m1[col].sum():,} trips ({df_m1[col].mean()*100:.1f}%)")


--- Before encoding ---
   payment_type values (M1): [1, 2]

✅ Encoding 1: payment_type → 2 columns: ['pay_1', 'pay_2']
   Distribution (M1):
      pay_1: 2,360,532 trips (86.7%)
      pay_2: 360,969 trips (13.3%)


In [110]:
# ============================================================
# ENCODING 2: RatecodeID → One-Hot
# ============================================================
# Original values:
#   1 = Standard rate (metered)
#   2 = JFK flat rate ($70)
#   3 = Newark flat rate
#   4 = Nassau/Westchester
#   5 = Negotiated fare
#   6 = Group ride
#
# WHY: Rate type directly determines how the fare is calculated.
# A JFK flat rate trip ($70) behaves completely differently from
# a standard metered trip, even if the distance is similar.
# The model needs to know which fare structure applies.
# ============================================================

print(f"\n--- Before encoding ---")
print(f"   RatecodeID values (M1): {sorted(df_m1['RatecodeID'].unique())}")

df_m1 = pd.get_dummies(df_m1, columns=['RatecodeID'], prefix='rate', dtype=int)
df_m2 = pd.get_dummies(df_m2, columns=['RatecodeID'], prefix='rate', dtype=int)

rate_cols = [c for c in df_m1.columns if c.startswith('rate_')]
print(f"\n✅ Encoding 2: RatecodeID → {len(rate_cols)} columns: {rate_cols}")
print(f"   Distribution (M1):")
for col in rate_cols:
    print(f"      {col}: {df_m1[col].sum():,} trips ({df_m1[col].mean()*100:.1f}%)")


--- Before encoding ---
   RatecodeID values (M1): [1.0, 2.0, 3.0, 4.0, 5.0]

✅ Encoding 2: RatecodeID → 5 columns: ['rate_1.0', 'rate_2.0', 'rate_3.0', 'rate_4.0', 'rate_5.0']
   Distribution (M1):
      rate_1.0: 2,612,018 trips (96.0%)
      rate_2.0: 85,061 trips (3.1%)
      rate_3.0: 6,805 trips (0.3%)
      rate_4.0: 6,192 trips (0.2%)
      rate_5.0: 11,425 trips (0.4%)


In [111]:
# ============================================================
# ENCODING 3: VendorID → One-Hot
# ============================================================
# Original values:
#   1 = Creative Mobile Technologies (CMT)
#   2 = VeriFone Inc.
#
# WHY: Different vendors use different meter technology.
# There may be subtle differences in how they record data
# or calibrate meters. The model can learn if one vendor's
# trips have slightly different patterns.
# ============================================================

print(f"\n--- Before encoding ---")
print(f"   VendorID values (M1): {sorted(df_m1['VendorID'].unique())}")

df_m1 = pd.get_dummies(df_m1, columns=['VendorID'], prefix='vendor', dtype=int)
df_m2 = pd.get_dummies(df_m2, columns=['VendorID'], prefix='vendor', dtype=int)

vendor_cols = [c for c in df_m1.columns if c.startswith('vendor_')]
print(f"\n✅ Encoding 3: VendorID → {len(vendor_cols)} columns: {vendor_cols}")
print(f"   Distribution (M1):")
for col in vendor_cols:
    print(f"      {col}: {df_m1[col].sum():,} trips ({df_m1[col].mean()*100:.1f}%)")


--- Before encoding ---
   VendorID values (M1): [1, 2]

✅ Encoding 3: VendorID → 2 columns: ['vendor_1', 'vendor_2']
   Distribution (M1):
      vendor_1: 579,841 trips (21.3%)
      vendor_2: 2,141,660 trips (78.7%)


In [112]:
# ============================================================
# ENCODING 4: time_period → One-Hot
# ============================================================
# Created in Phase 1:
#   'early_morning' (0-5 AM)
#   'morning' (6-11 AM)
#   'afternoon' (12-4 PM)
#   'evening' (5-8 PM)
#   'night' (9-11 PM)
#
# WHY: time_period is currently a text category. The model
# can't use "morning" as input — it needs numbers.
# After encoding, each time block becomes its own column.
# ============================================================

print(f"\n--- Before encoding ---")
print(f"   time_period values (M1): {df_m1['time_period'].unique().tolist()}")

df_m1 = pd.get_dummies(df_m1, columns=['time_period'], prefix='period', dtype=int)
df_m2 = pd.get_dummies(df_m2, columns=['time_period'], prefix='period', dtype=int)

period_cols = [c for c in df_m1.columns if c.startswith('period_')]
print(f"\n✅ Encoding 4: time_period → {len(period_cols)} columns: {period_cols}")
print(f"   Distribution (M1):")
for col in period_cols:
    print(f"      {col}: {df_m1[col].sum():,} trips ({df_m1[col].mean()*100:.1f}%)")


--- Before encoding ---
   time_period values (M1): ['early_morning', 'night', 'evening', 'morning', 'afternoon']

✅ Encoding 4: time_period → 5 columns: ['period_early_morning', 'period_morning', 'period_afternoon', 'period_evening', 'period_night']
   Distribution (M1):
      period_early_morning: 177,212 trips (6.5%)
      period_morning: 586,409 trips (21.5%)
      period_afternoon: 849,947 trips (31.2%)
      period_evening: 727,054 trips (26.7%)
      period_night: 380,879 trips (14.0%)


In [113]:
# ============================================================
# INTERACTION FEATURE 1: distance_x_passenger
# ============================================================
# Formula: trip_distance × passenger_count
#
# WHY THIS INTERACTION MATTERS:
# Distance and passenger count TOGETHER tell a richer story
# than each one alone:
#
# - 10 miles × 1 passenger = 10 → solo commuter on highway
# - 10 miles × 4 passengers = 40 → family going to airport
# - 2 miles × 1 passenger = 2 → quick solo errand
# - 2 miles × 4 passengers = 8 → group bar-hopping
#
# The combined value captures trip "scale" — a long trip
# with many passengers is a fundamentally different event
# than a short solo ride, even if the individual values
# of distance or passengers don't seem extreme.
#
# USED IN: Both models (both values known at pickup)
# ============================================================

df_m1['distance_x_passenger'] = df_m1['trip_distance'] * df_m1['passenger_count']
df_m2['distance_x_passenger'] = df_m2['trip_distance'] * df_m2['passenger_count']

print(f"\n✅ Interaction 1: distance_x_passenger")
print(f"   Mean (M1): {df_m1['distance_x_passenger'].mean():.2f}")
print(f"   Median (M1): {df_m1['distance_x_passenger'].median():.2f}")
print(f"   Range (M1): {df_m1['distance_x_passenger'].min():.2f} to {df_m1['distance_x_passenger'].max():.2f}")


✅ Interaction 1: distance_x_passenger
   Mean (M1): 4.21
   Median (M1): 1.90
   Range (M1): 0.01 to 389.68


In [114]:
# ============================================================
# INTERACTION FEATURE 2: hour_x_day_of_week
# ============================================================
# Formula: pickup_hour × pickup_day_of_week
#
# WHY THIS INTERACTION MATTERS:
# Time and day TOGETHER create unique traffic patterns:
#
# - Hour 8 × Day 0 (Monday) = 0 → Monday morning rush
# - Hour 8 × Day 6 (Sunday) = 48 → quiet Sunday morning
# - Hour 22 × Day 4 (Friday) = 88 → Friday night out
# - Hour 22 × Day 1 (Tuesday) = 22 → quiet Tuesday night
#
# The same hour behaves very differently depending on
# which day it is. 8 AM Monday is NYC's worst congestion,
# but 8 AM Sunday is practically empty streets.
#
# The model gets pickup_hour and pickup_day_of_week
# separately from Phase 1, but this interaction helps it
# learn SPECIFIC time-day combinations without having to
# figure out the multiplication itself.
#
# USED IN: Both models (both values known at pickup)
# ============================================================

df_m1['hour_x_day_of_week'] = df_m1['pickup_hour'] * df_m1['pickup_day_of_week']
df_m2['hour_x_day_of_week'] = df_m2['pickup_hour'] * df_m2['pickup_day_of_week']

print(f"\n✅ Interaction 2: hour_x_day_of_week")
print(f"   Mean (M1): {df_m1['hour_x_day_of_week'].mean():.2f}")
print(f"   Range (M1): {df_m1['hour_x_day_of_week'].min()} to {df_m1['hour_x_day_of_week'].max()}")
print(f"   Example values:")
print(f"      Monday 8AM:  8 × 0 = {8*0}")
print(f"      Friday 6PM: 18 × 4 = {18*4}")
print(f"      Sunday 8AM:  8 × 6 = {8*6}")


✅ Interaction 2: hour_x_day_of_week
   Mean (M1): 43.12
   Range (M1): 0 to 138
   Example values:
      Monday 8AM:  8 × 0 = 0
      Friday 6PM: 18 × 4 = 72
      Sunday 8AM:  8 × 6 = 48


In [115]:
# ============================================================
# PHASE 4 SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PHASE 4 COMPLETE: CATEGORICAL ENCODING & INTERACTION FEATURES")
print("=" * 70)

# Count all encoded columns
all_encoded = pay_cols + rate_cols + vendor_cols + period_cols
interaction_features = ['distance_x_passenger', 'hour_x_day_of_week']

print(f"\n✅ One-hot encoded columns created: {len(all_encoded)}")
print(f"   payment_type → {len(pay_cols)} columns")
print(f"   RatecodeID → {len(rate_cols)} columns")
print(f"   VendorID → {len(vendor_cols)} columns")
print(f"   time_period → {len(period_cols)} columns")

print(f"\n✅ Interaction features created: {len(interaction_features)}")
for f in interaction_features:
    print(f"   • {f}")

print(f"\n   Total new columns from Phase 4: {len(all_encoded) + len(interaction_features)}")

# Note: Original columns (payment_type, RatecodeID, VendorID, time_period)
# are automatically REMOVED by pd.get_dummies — no manual dropping needed!
print(f"\n   ℹ️ Original categorical columns (payment_type, RatecodeID,")
print(f"      VendorID, time_period) have been automatically removed")
print(f"      by pd.get_dummies and replaced with their encoded versions.")

print(f"\nModel 1 shape: {df_m1.shape[0]:,} rows × {df_m1.shape[1]} columns (was 54 after Phase 3)")
print(f"Model 2 shape: {df_m2.shape[0]:,} rows × {df_m2.shape[1]} columns (was 54 after Phase 3)")


PHASE 4 COMPLETE: CATEGORICAL ENCODING & INTERACTION FEATURES

✅ One-hot encoded columns created: 14
   payment_type → 2 columns
   RatecodeID → 5 columns
   VendorID → 2 columns
   time_period → 5 columns

✅ Interaction features created: 2
   • distance_x_passenger
   • hour_x_day_of_week

   Total new columns from Phase 4: 16

   ℹ️ Original categorical columns (payment_type, RatecodeID,
      VendorID, time_period) have been automatically removed
      by pd.get_dummies and replaced with their encoded versions.

Model 1 shape: 2,721,501 rows × 66 columns (was 54 after Phase 3)
Model 2 shape: 2,412,426 rows × 66 columns (was 54 after Phase 3)


# Phase 5

In [116]:
# ============================================================
# ============================================================
#
#   PHASE 5: FEATURE SELECTION & COLUMN CLEANUP
#   Removing unnecessary columns & encoding remaining categories
#
# ============================================================
# ============================================================
# WHY CLEANUP MATTERS:
# Right now our dataframes have 66 columns — but not all of
# them should go into the model:
#
# - Some are raw columns we've already extracted info from
#   (like datetime — we got hour, day, weekend from it)
# - Some would cause data leakage (like cbd_congestion_fee
#   in Model 2 — it directly reveals the answer!)
# - Some are analysis-only (speed, fare_per_mile etc.)
# - Some categorical columns still need encoding
#
# Think of it like packing for a trip:
# - Keep what you need (modeling features)
# - Set aside souvenirs (analysis features — useful but separate)
# - Leave behind what doesn't belong (raw columns, leaky features)
# ============================================================


print("\n" + "=" * 70)
print("PHASE 5: FEATURE SELECTION & COLUMN CLEANUP")
print("=" * 70)

print(f"\nStarting columns: M1 = {df_m1.shape[1]}, M2 = {df_m2.shape[1]}")


PHASE 5: FEATURE SELECTION & COLUMN CLEANUP

Starting columns: M1 = 66, M2 = 66


In [117]:
# ============================================================
# STEP 1: ENCODE REMAINING CATEGORICAL COLUMNS
# ============================================================
# These categorical features were created in Phases 2 and 3
# but not yet encoded in Phase 4. We encode them now before
# dropping any columns.
#
# Columns to encode:
#   - distance_category (Phase 3: short/medium/long/very_long)
#   - cbd_trip_type (Phase 2: within_cbd/into_cbd/out_of_cbd/outside_cbd)
#   - passenger_count_binned (Phase 3: 1/2/3+)
#   - pu_zone_cluster (Phase 2: low/medium/high)
#   - do_zone_cluster (Phase 2: low/medium/high)
# ============================================================

print(f"\n--- Step 1: Encode remaining categorical columns ---")

# distance_category
print(f"\n   Before: distance_category values = {df_m1['distance_category'].unique().tolist()}")
df_m1 = pd.get_dummies(df_m1, columns=['distance_category'], prefix='dist', dtype=int)
df_m2 = pd.get_dummies(df_m2, columns=['distance_category'], prefix='dist', dtype=int)
dist_cols = [c for c in df_m1.columns if c.startswith('dist_')]
print(f"   ✅ distance_category → {len(dist_cols)} columns: {dist_cols}")

# cbd_trip_type
print(f"\n   Before: cbd_trip_type values = {df_m1['cbd_trip_type'].unique().tolist()}")
df_m1 = pd.get_dummies(df_m1, columns=['cbd_trip_type'], prefix='cbd', dtype=int)
df_m2 = pd.get_dummies(df_m2, columns=['cbd_trip_type'], prefix='cbd', dtype=int)
cbd_cols = [c for c in df_m1.columns if c.startswith('cbd_')]
print(f"   ✅ cbd_trip_type → {len(cbd_cols)} columns: {cbd_cols}")

# passenger_count_binned
print(f"\n   Before: passenger_count_binned values = {df_m1['passenger_count_binned'].unique().tolist()}")
df_m1 = pd.get_dummies(df_m1, columns=['passenger_count_binned'], prefix='pax', dtype=int)
df_m2 = pd.get_dummies(df_m2, columns=['passenger_count_binned'], prefix='pax', dtype=int)
pax_cols = [c for c in df_m1.columns if c.startswith('pax_')]
print(f"   ✅ passenger_count_binned → {len(pax_cols)} columns: {pax_cols}")

# pu_zone_cluster
print(f"\n   Before: pu_zone_cluster values = {df_m1['pu_zone_cluster'].unique().tolist()}")
df_m1 = pd.get_dummies(df_m1, columns=['pu_zone_cluster'], prefix='pu_cluster', dtype=int)
df_m2 = pd.get_dummies(df_m2, columns=['pu_zone_cluster'], prefix='pu_cluster', dtype=int)
pu_cluster_cols = [c for c in df_m1.columns if c.startswith('pu_cluster_')]
print(f"   ✅ pu_zone_cluster → {len(pu_cluster_cols)} columns: {pu_cluster_cols}")

# do_zone_cluster
print(f"\n   Before: do_zone_cluster values = {df_m1['do_zone_cluster'].unique().tolist()}")
df_m1 = pd.get_dummies(df_m1, columns=['do_zone_cluster'], prefix='do_cluster', dtype=int)
df_m2 = pd.get_dummies(df_m2, columns=['do_zone_cluster'], prefix='do_cluster', dtype=int)
do_cluster_cols = [c for c in df_m1.columns if c.startswith('do_cluster_')]
print(f"   ✅ do_zone_cluster → {len(do_cluster_cols)} columns: {do_cluster_cols}")

total_new_encoded = len(dist_cols) + len(cbd_cols) + len(pax_cols) + len(pu_cluster_cols) + len(do_cluster_cols)
print(f"\n   Total new encoded columns: {total_new_encoded}")
print(f"   Columns after encoding: M1 = {df_m1.shape[1]}, M2 = {df_m2.shape[1]}")


--- Step 1: Encode remaining categorical columns ---

   Before: distance_category values = ['short', 'medium', 'long', 'very_long']
   ✅ distance_category → 4 columns: ['dist_short', 'dist_medium', 'dist_long', 'dist_very_long']

   Before: cbd_trip_type values = ['out_of_cbd', 'outside_cbd', 'into_cbd', 'within_cbd']
   ✅ cbd_trip_type → 5 columns: ['cbd_congestion_fee', 'cbd_into_cbd', 'cbd_out_of_cbd', 'cbd_outside_cbd', 'cbd_within_cbd']

   Before: passenger_count_binned values = ['1', '3+', '2']
   ✅ passenger_count_binned → 3 columns: ['pax_1', 'pax_2', 'pax_3+']

   Before: pu_zone_cluster values = ['high', 'medium', 'low']
   ✅ pu_zone_cluster → 3 columns: ['pu_cluster_high', 'pu_cluster_low', 'pu_cluster_medium']

   Before: do_zone_cluster values = ['high', 'medium', 'low']
   ✅ do_zone_cluster → 3 columns: ['do_cluster_high', 'do_cluster_low', 'do_cluster_medium']

   Total new encoded columns: 18
   Columns after encoding: M1 = 78, M2 = 78


In [118]:
# ============================================================
# STEP 2: SEPARATE ANALYSIS-ONLY FEATURES
# ============================================================
# These 4 features use post-trip information and CANNOT be
# used in models. We save them to a separate dataframe so:
# 1. They're preserved for stakeholder analysis
# 2. They can't accidentally enter the model
#
# Think of it like putting fragile items in a separate box
# when you're moving — they're valuable but need to be
# handled differently.
# ============================================================

print(f"\n--- Step 2: Separate analysis-only features ---")

analysis_cols = ['avg_speed_mph', 'fare_per_mile', 'tip_percentage', 'cost_per_minute']

# Save analysis features with key identifiers for later use
# Include PULocationID, pickup_hour, and targets for analysis grouping
analysis_id_cols = ['PULocationID', 'DOLocationID', 'pickup_hour', 'pickup_day_of_week']

# Create analysis dataframes
df_analysis_m1 = df_m1[analysis_id_cols + analysis_cols + ['trip_duration_min']].copy()
df_analysis_m2 = df_m2[analysis_id_cols + analysis_cols + ['has_congestion_fee']].copy()

# Drop analysis columns from modeling dataframes
df_m1 = df_m1.drop(columns=analysis_cols)
df_m2 = df_m2.drop(columns=analysis_cols)

print(f"   ✅ Analysis features separated into df_analysis_m1 and df_analysis_m2")
print(f"   Analysis features removed from modeling data: {analysis_cols}")
print(f"   Analysis M1 shape: {df_analysis_m1.shape}")
print(f"   Analysis M2 shape: {df_analysis_m2.shape}")
print(f"   Modeling columns after removal: M1 = {df_m1.shape[1]}, M2 = {df_m2.shape[1]}")


--- Step 2: Separate analysis-only features ---
   ✅ Analysis features separated into df_analysis_m1 and df_analysis_m2
   Analysis features removed from modeling data: ['avg_speed_mph', 'fare_per_mile', 'tip_percentage', 'cost_per_minute']
   Analysis M1 shape: (2721501, 9)
   Analysis M2 shape: (2412426, 9)
   Modeling columns after removal: M1 = 74, M2 = 74


In [119]:
# ============================================================
# STEP 3: DROP RAW DATETIME COLUMNS
# ============================================================
# We've extracted all useful information from these in Phase 1:
#   - pickup hour, day of week, day of month
#   - is_weekend, is_rush_hour, is_peak_hour
#   - time_period (now encoded)
#   - cyclical encodings (sin/cos)
#
# The raw datetime objects can't be used by ML models anyway.
# Keeping them would just waste memory.
# ============================================================

print(f"\n--- Step 3: Drop raw datetime columns ---")

datetime_cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime']

df_m1 = df_m1.drop(columns=datetime_cols)
df_m2 = df_m2.drop(columns=datetime_cols)

print(f"   ✅ Dropped: {datetime_cols}")
print(f"   Columns after drop: M1 = {df_m1.shape[1]}, M2 = {df_m2.shape[1]}")


--- Step 3: Drop raw datetime columns ---
   ✅ Dropped: ['tpep_pickup_datetime', 'tpep_dropoff_datetime']
   Columns after drop: M1 = 72, M2 = 72


In [120]:
# ============================================================
# STEP 4: DROP store_and_fwd_flag
# ============================================================
# This flag indicates if the trip record was held in vehicle
# memory before sending to the server (Y/N).
# It's almost always 'N' (99.9%+ of trips) — near-zero variance.
# A feature with almost no variation provides no useful signal.
#
# ANALOGY: Imagine a survey question where 99.9% of people
# answer "No." That question tells you nothing useful about
# the differences between people.
# ============================================================

print(f"\n--- Step 4: Drop store_and_fwd_flag ---")

# Check the distribution first
print(f"   store_and_fwd_flag distribution (M1):")
print(f"   {df_m1['store_and_fwd_flag'].value_counts().to_string()}")

df_m1 = df_m1.drop(columns=['store_and_fwd_flag'])
df_m2 = df_m2.drop(columns=['store_and_fwd_flag'])

print(f"   ✅ Dropped: store_and_fwd_flag (near-zero variance)")
print(f"   Columns after drop: M1 = {df_m1.shape[1]}, M2 = {df_m2.shape[1]}")


--- Step 4: Drop store_and_fwd_flag ---
   store_and_fwd_flag distribution (M1):
   store_and_fwd_flag
N    2714765
Y       6736
   ✅ Dropped: store_and_fwd_flag (near-zero variance)
   Columns after drop: M1 = 71, M2 = 71


In [121]:
# ============================================================
# STEP 5: DROP LEAKY COLUMNS FROM MODEL 2
# ============================================================
# For Model 2 (congestion fee prediction), we must remove
# columns that directly reveal or strongly leak the target:
#
# cbd_congestion_fee: This IS the target in dollar form!
#   has_congestion_fee was derived FROM this column.
#   $0.75 → 1, $0.00 → 0. Keeping it = instant 100% accuracy
#   but completely useless in the real world.
#
# congestion_surcharge: This is a separate surcharge that
#   also relates to congestion pricing — could leak info.
#
# Fare-related columns for Model 2:
#   fare_amount, extra, mta_tax, tip_amount, tolls_amount,
#   improvement_surcharge, total_amount, Airport_fee
#   These are all POST-TRIP values. Maria doesn't know them
#   when she accepts the ride.
#
# trip_duration_min: Also a post-trip value — Maria doesn't
#   know how long the trip will take before it starts.
#   Removed in the cross-model fix below.
#
# For Model 1 (duration prediction):
#   We also drop fare-related columns since they're determined
#   AFTER the trip (the meter runs during the trip).
#   We keep cbd_congestion_fee and congestion_surcharge as
#   they don't leak duration information.
# ============================================================

print(f"\n--- Step 5: Drop leaky/post-trip columns ---")

# Columns that are post-trip for BOTH models
post_trip_cols = [
    'fare_amount', 'extra', 'mta_tax', 'tip_amount',
    'tolls_amount', 'improvement_surcharge', 'total_amount'
]

# Model 1: Drop post-trip fare columns
# Keep congestion-related columns (they don't leak duration)
df_m1 = df_m1.drop(columns=post_trip_cols)
print(f"   ✅ Model 1 — Dropped post-trip fare columns: {len(post_trip_cols)} columns")

# Model 2: Drop post-trip fare columns AND congestion fee (direct leakage)
m2_drop_cols = post_trip_cols + ['cbd_congestion_fee', 'congestion_surcharge', 'Airport_fee']
df_m2 = df_m2.drop(columns=m2_drop_cols)
print(f"   ✅ Model 2 — Dropped post-trip + leaky columns: {len(m2_drop_cols)} columns")
print(f"      Including cbd_congestion_fee (directly reveals target!)")

print(f"   Columns after drop: M1 = {df_m1.shape[1]}, M2 = {df_m2.shape[1]}")


--- Step 5: Drop leaky/post-trip columns ---
   ✅ Model 1 — Dropped post-trip fare columns: 7 columns
   ✅ Model 2 — Dropped post-trip + leaky columns: 10 columns
      Including cbd_congestion_fee (directly reveals target!)
   Columns after drop: M1 = 64, M2 = 61


In [122]:
# ============================================================
# STEP 6: VERIFY NO DATA LEAKAGE REMAINS
# ============================================================
# Final safety check — make sure no column in the modeling
# data could leak the target variable.
#
# For Model 1: target is trip_duration_min
#   → No other column should contain duration info
#
# For Model 2: target is has_congestion_fee
#   → No column should directly reveal whether fee was charged
# ============================================================

print(f"\n--- Step 6: Data leakage safety check ---")

# Model 1: Check that target-related columns are only the target itself
print(f"\n   Model 1 target: trip_duration_min")
suspicious_m1 = [c for c in df_m1.columns if 'duration' in c.lower() and c != 'trip_duration_min']
if suspicious_m1:
    print(f"   ⚠️ Suspicious columns found: {suspicious_m1}")
else:
    print(f"   ✅ No duration-leaking columns found (besides target)")

# Model 2: Check that target-related columns are only the target itself
print(f"\n   Model 2 target: has_congestion_fee")
suspicious_m2 = [c for c in df_m2.columns if 'congestion' in c.lower() or 'cbd_congestion' in c.lower()]
if suspicious_m2:
    # cbd_into_cbd, cbd_outside_cbd etc. are FINE — they're geographic features
    geo_cbd = [c for c in suspicious_m2 if c.startswith('cbd_')]
    non_geo = [c for c in suspicious_m2 if not c.startswith('cbd_')]
    if non_geo:
        print(f"   ⚠️ Potentially leaky columns: {non_geo}")
    else:
        print(f"   ✅ CBD geographic features present (expected): {geo_cbd}")
        print(f"   ✅ No congestion fee leaking columns found (besides target)")
else:
    print(f"   ✅ No congestion-leaking columns found")


--- Step 6: Data leakage safety check ---

   Model 1 target: trip_duration_min
   ✅ No duration-leaking columns found (besides target)

   Model 2 target: has_congestion_fee
   ⚠️ Potentially leaky columns: ['has_congestion_fee']


In [123]:
# ============================================================
# FIX: Remove cross-model target variables
# ============================================================
# Model 1 doesn't need has_congestion_fee (it predicts duration)
# Model 2 doesn't need trip_duration_min (it predicts fee, 
#   and duration isn't known before the trip)
# ============================================================

print("--- Fix: Remove cross-model targets ---")

# Model 1: drop congestion fee target (not relevant to duration prediction)
df_m1 = df_m1.drop(columns=['has_congestion_fee'])
print(f"   ✅ Model 1: Dropped 'has_congestion_fee' (not needed for duration prediction)")

# Model 2: drop duration (data leakage — not known before trip)
df_m2 = df_m2.drop(columns=['trip_duration_min'])
print(f"   ✅ Model 2: Dropped 'trip_duration_min' (not known before trip — leakage!)")

print(f"\n   Model 1: {df_m1.shape[0]:,} rows × {df_m1.shape[1]} columns")
print(f"   Model 2: {df_m2.shape[0]:,} rows × {df_m2.shape[1]} columns")

--- Fix: Remove cross-model targets ---
   ✅ Model 1: Dropped 'has_congestion_fee' (not needed for duration prediction)
   ✅ Model 2: Dropped 'trip_duration_min' (not known before trip — leakage!)

   Model 1: 2,721,501 rows × 63 columns
   Model 2: 2,412,426 rows × 60 columns


In [124]:
# ============================================================
# STEP 6: VERIFY NO DATA LEAKAGE REMAINS
# ============================================================
# Final safety check — make sure no column in the modeling
# data could leak the target variable.
#
# For Model 1: target is trip_duration_min
#   → No other column should contain duration info
#
# For Model 2: target is has_congestion_fee
#   → No column should directly reveal whether fee was charged
# ============================================================

print(f"\n--- Step 6: Data leakage safety check ---")

# Model 1: Check that target-related columns are only the target itself
print(f"\n   Model 1 target: trip_duration_min")
suspicious_m1 = [c for c in df_m1.columns if 'duration' in c.lower() and c != 'trip_duration_min']
if suspicious_m1:
    print(f"   ⚠️ Suspicious columns found: {suspicious_m1}")
else:
    print(f"   ✅ No duration-leaking columns found (besides target)")

# Model 2: Check that target-related columns are only the target itself
print(f"\n   Model 2 target: has_congestion_fee")
suspicious_m2 = [c for c in df_m2.columns if ('congestion' in c.lower() or 'cbd_congestion' in c.lower()) and c != 'has_congestion_fee']
if suspicious_m2:
    # cbd_into_cbd, cbd_outside_cbd etc. are FINE — they're geographic features
    geo_cbd = [c for c in suspicious_m2 if c.startswith('cbd_')]
    non_geo = [c for c in suspicious_m2 if not c.startswith('cbd_')]
    if non_geo:
        print(f"   ⚠️ Potentially leaky columns: {non_geo}")
    else:
        print(f"   ✅ CBD geographic features present (expected): {geo_cbd}")
        print(f"   ✅ No congestion fee leaking columns found (besides target)")
else:
    print(f"   ✅ No congestion-leaking columns found")


--- Step 6: Data leakage safety check ---

   Model 1 target: trip_duration_min
   ✅ No duration-leaking columns found (besides target)

   Model 2 target: has_congestion_fee
   ✅ No congestion-leaking columns found


In [125]:
# ============================================================
# PHASE 5 SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("PHASE 5 COMPLETE: FEATURE SELECTION & COLUMN CLEANUP")
print("=" * 70)

print(f"\n📋 What we did:")
print(f"   1. Encoded 5 remaining categorical columns → {total_new_encoded} new binary columns")
print(f"   2. Separated 4 analysis-only features into dedicated dataframes")
print(f"   3. Dropped 2 raw datetime columns")
print(f"   4. Dropped store_and_fwd_flag (near-zero variance)")
print(f"   5. Dropped post-trip fare columns (data leakage prevention)")
print(f"   6. Verified no data leakage remains")

print(f"\nModel 1 shape: {df_m1.shape[0]:,} rows × {df_m1.shape[1]} columns")
print(f"Model 2 shape: {df_m2.shape[0]:,} rows × {df_m2.shape[1]} columns")
print(f"Analysis M1: {df_analysis_m1.shape[0]:,} rows × {df_analysis_m1.shape[1]} columns")
print(f"Analysis M2: {df_analysis_m2.shape[0]:,} rows × {df_analysis_m2.shape[1]} columns")

print(f"\n📋 Final column lists:")
print(f"\n   Model 1 columns ({df_m1.shape[1]}):")
for i, col in enumerate(sorted(df_m1.columns), 1):
    print(f"      {i:2d}. {col}")

print(f"\n   Model 2 columns ({df_m2.shape[1]}):")
for i, col in enumerate(sorted(df_m2.columns), 1):
    print(f"      {i:2d}. {col}")


PHASE 5 COMPLETE: FEATURE SELECTION & COLUMN CLEANUP

📋 What we did:
   1. Encoded 5 remaining categorical columns → 18 new binary columns
   2. Separated 4 analysis-only features into dedicated dataframes
   3. Dropped 2 raw datetime columns
   4. Dropped store_and_fwd_flag (near-zero variance)
   5. Dropped post-trip fare columns (data leakage prevention)
   6. Verified no data leakage remains

Model 1 shape: 2,721,501 rows × 63 columns
Model 2 shape: 2,412,426 rows × 60 columns
Analysis M1: 2,721,501 rows × 9 columns
Analysis M2: 2,412,426 rows × 9 columns

📋 Final column lists:

   Model 1 columns (63):
       1. Airport_fee
       2. DOLocationID
       3. PULocationID
       4. cbd_congestion_fee
       5. cbd_into_cbd
       6. cbd_out_of_cbd
       7. cbd_outside_cbd
       8. cbd_within_cbd
       9. congestion_surcharge
      10. dist_long
      11. dist_medium
      12. dist_short
      13. dist_very_long
      14. distance_x_passenger
      15. do_cluster_high
      16. do

# Phase 6

In [126]:
# ============================================================
# ============================================================
#
#   PHASE 6: VERIFICATION & SAVE
#   Final quality checks and saving feature-engineered datasets
#
# ============================================================
# ============================================================
# WHY VERIFICATION MATTERS:
# Before saving, we do one last thorough check — like a pilot
# going through a pre-flight checklist before takeoff.
# If something is wrong, it's much easier to fix now than
# to discover it during modeling when results don't make sense.
#
# We check for:
# 1. Null values (features with missing data)
# 2. Infinite values (from division operations)
# 3. Data types (all columns should be numeric for ML)
# 4. Column counts match expectations
# 5. Target variables are present and correct
# 6. Row counts haven't changed (no accidental data loss)
# ============================================================


print("\n" + "=" * 70)
print("PHASE 6: VERIFICATION & SAVE")
print("=" * 70)


PHASE 6: VERIFICATION & SAVE


In [127]:
# ============================================================
# CHECK 1: NULL VALUES
# ============================================================
# Feature engineering can introduce nulls through:
# - Division by zero (we handled in Phase 3)
# - Mapping failures (zone not found in lookup)
# - Encoding edge cases
# Any nulls here would cause models to fail or perform poorly.
# ============================================================

print(f"\n--- Check 1: Null values ---")

nulls_m1 = df_m1.isnull().sum()
nulls_m2 = df_m2.isnull().sum()

total_nulls_m1 = nulls_m1.sum()
total_nulls_m2 = nulls_m2.sum()

if total_nulls_m1 == 0:
    print(f"   ✅ Model 1: No null values in any column")
else:
    print(f"   ⚠️ Model 1: {total_nulls_m1:,} null values found:")
    print(nulls_m1[nulls_m1 > 0].to_string())

if total_nulls_m2 == 0:
    print(f"   ✅ Model 2: No null values in any column")
else:
    print(f"   ⚠️ Model 2: {total_nulls_m2:,} null values found:")
    print(nulls_m2[nulls_m2 > 0].to_string())


--- Check 1: Null values ---
   ✅ Model 1: No null values in any column
   ✅ Model 2: No null values in any column


In [128]:
# ============================================================
# CHECK 2: INFINITE VALUES
# ============================================================
# Division features could still have sneaky infinities.
# ML models can't handle infinity — they'll crash or
# produce garbage predictions.
# ============================================================

print(f"\n--- Check 2: Infinite values ---")

inf_m1 = np.isinf(df_m1.select_dtypes(include=[np.number])).sum().sum()
inf_m2 = np.isinf(df_m2.select_dtypes(include=[np.number])).sum().sum()

if inf_m1 == 0:
    print(f"   ✅ Model 1: No infinite values")
else:
    print(f"   ⚠️ Model 1: {inf_m1:,} infinite values found")

if inf_m2 == 0:
    print(f"   ✅ Model 2: No infinite values")
else:
    print(f"   ⚠️ Model 2: {inf_m2:,} infinite values found")


--- Check 2: Infinite values ---
   ✅ Model 1: No infinite values
   ✅ Model 2: No infinite values


In [129]:
# ============================================================
# CHECK 3: DATA TYPES
# ============================================================
# All columns should be numeric (int or float) for ML models.
# If any text/object columns remain, models will fail.
# ============================================================

print(f"\n--- Check 3: Data types ---")

non_numeric_m1 = df_m1.select_dtypes(exclude=[np.number]).columns.tolist()
non_numeric_m2 = df_m2.select_dtypes(exclude=[np.number]).columns.tolist()

if len(non_numeric_m1) == 0:
    print(f"   ✅ Model 1: All {df_m1.shape[1]} columns are numeric")
else:
    print(f"   ⚠️ Model 1: Non-numeric columns found: {non_numeric_m1}")

if len(non_numeric_m2) == 0:
    print(f"   ✅ Model 2: All {df_m2.shape[1]} columns are numeric")
else:
    print(f"   ⚠️ Model 2: Non-numeric columns found: {non_numeric_m2}")

print(f"\n   Model 1 data types:")
print(f"   {df_m1.dtypes.value_counts().to_string()}")
print(f"\n   Model 2 data types:")
print(f"   {df_m2.dtypes.value_counts().to_string()}")


--- Check 3: Data types ---
   ✅ Model 1: All 63 columns are numeric
   ✅ Model 2: All 60 columns are numeric

   Model 1 data types:
   int32      49
float64    12
int64       2

   Model 2 data types:
   int32      50
float64     8
int64       2


In [130]:
# ============================================================
# CHECK 4: COLUMN COUNTS
# ============================================================
# Verify we have the expected number of columns.
# Model 1: 63 columns (62 features + 1 target)
# Model 2: 60 columns (59 features + 1 target)
# ============================================================

print(f"\n--- Check 4: Column counts ---")
print(f"   Model 1: {df_m1.shape[1]} columns (expected: 63)")
print(f"   Model 2: {df_m2.shape[1]} columns (expected: 60)")

if df_m1.shape[1] == 63 and df_m2.shape[1] == 60:
    print(f"   ✅ Column counts match expectations!")
else:
    print(f"   ⚠️ Column counts differ from expected — review Phase 5")


--- Check 4: Column counts ---
   Model 1: 63 columns (expected: 63)
   Model 2: 60 columns (expected: 60)
   ✅ Column counts match expectations!


In [131]:
# ============================================================
# CHECK 5: TARGET VARIABLES
# ============================================================
# Make sure each model has its target and the target looks right.
# Model 1 target: trip_duration_min (continuous, 1-179 minutes)
# Model 2 target: has_congestion_fee (binary, 0 or 1)
# ============================================================

print(f"\n--- Check 5: Target variables ---")

# Model 1 target
print(f"\n   Model 1 target: trip_duration_min")
print(f"      Present: {'trip_duration_min' in df_m1.columns}")
print(f"      Range: {df_m1['trip_duration_min'].min():.2f} to {df_m1['trip_duration_min'].max():.2f} min")
print(f"      Mean: {df_m1['trip_duration_min'].mean():.2f} min")
print(f"      Median: {df_m1['trip_duration_min'].median():.2f} min")

# Model 2 target
print(f"\n   Model 2 target: has_congestion_fee")
print(f"      Present: {'has_congestion_fee' in df_m2.columns}")
print(f"      Distribution:")
print(f"         1 (fee charged): {(df_m2['has_congestion_fee'] == 1).sum():,} ({df_m2['has_congestion_fee'].mean()*100:.1f}%)")
print(f"         0 (no fee):      {(df_m2['has_congestion_fee'] == 0).sum():,} ({(1 - df_m2['has_congestion_fee'].mean())*100:.1f}%)")


--- Check 5: Target variables ---

   Model 1 target: trip_duration_min
      Present: True
      Range: 1.00 to 179.03 min
      Mean: 14.17 min
      Median: 11.25 min

   Model 2 target: has_congestion_fee
      Present: True
      Distribution:
         1 (fee charged): 1,798,485 (74.6%)
         0 (no fee):      613,941 (25.4%)


In [132]:
# ============================================================
# CHECK 6: ROW COUNTS
# ============================================================
# Verify no rows were accidentally lost during feature engineering.
# Model 1 should still have 2,721,501 rows
# Model 2 should still have 2,412,426 rows
# ============================================================

print(f"\n--- Check 6: Row counts ---")
print(f"   Model 1: {df_m1.shape[0]:,} rows (expected: 2,721,501)")
print(f"   Model 2: {df_m2.shape[0]:,} rows (expected: 2,412,426)")

if df_m1.shape[0] == 2721501 and df_m2.shape[0] == 2412426:
    print(f"   ✅ Row counts match — no data lost during feature engineering!")
else:
    print(f"   ⚠️ Row counts differ — some rows may have been dropped")


--- Check 6: Row counts ---
   Model 1: 2,721,501 rows (expected: 2,721,501)
   Model 2: 2,412,426 rows (expected: 2,412,426)
   ✅ Row counts match — no data lost during feature engineering!


In [133]:
# ============================================================
# CHECK 7: QUICK SAMPLE VERIFICATION
# ============================================================
# Look at a few sample rows to make sure values make sense.
# ============================================================

In [134]:
print(f"\n--- Check 7: Sample data verification ---")
print(f"\n   Model 1 — First row sample:")
print(df_m1.iloc[0].to_string())


--- Check 7: Sample data verification ---

   Model 1 — First row sample:
passenger_count                1.00
trip_distance                  1.60
PULocationID                 229.00
DOLocationID                 237.00
congestion_surcharge           2.50
Airport_fee                    0.00
cbd_congestion_fee             0.00
trip_duration_min              8.35
pickup_hour                    0.00
pickup_day_of_week             2.00
pickup_day_of_month            1.00
is_weekend                     0.00
is_rush_hour                   0.00
is_peak_hour                   0.00
pickup_hour_sin                0.00
pickup_hour_cos                1.00
pickup_dow_sin                 0.97
pickup_dow_cos                -0.22
is_same_zone                   0.00
pu_is_airport                  0.00
do_is_airport                  0.00
is_airport_trip                0.00
pu_is_cbd                      1.00
do_is_cbd                      0.00
pickup_zone_popularity     48709.00
dropoff_zone_popularity  

In [135]:
print(f"\n--- Check 7: Sample data verification ---")
print(f"\n   Model 2 — First row sample:")
print(df_m2.iloc[0].to_string())


--- Check 7: Sample data verification ---

   Model 2 — First row sample:
passenger_count               1.00
trip_distance                 8.66
PULocationID                138.00
DOLocationID                 43.00
has_congestion_fee            0.00
pickup_hour                   0.00
pickup_day_of_week            6.00
pickup_day_of_month           5.00
is_weekend                    1.00
is_rush_hour                  0.00
is_peak_hour                  0.00
pickup_hour_sin               0.00
pickup_hour_cos               1.00
pickup_dow_sin               -0.78
pickup_dow_cos                0.62
is_same_zone                  0.00
pu_is_airport                 1.00
do_is_airport                 0.00
is_airport_trip               1.00
pu_is_cbd                     0.00
do_is_cbd                     1.00
pickup_zone_popularity    73114.00
dropoff_zone_popularity   27089.00
log_trip_distance             2.27
is_single_passenger           1.00
is_extreme_distance           0.00
is_extreme_fare

In [136]:
# ============================================================
# SAVE FEATURE-ENGINEERED DATASETS
# ============================================================
# Save 4 files:
# 1. fe_model1_duration.parquet — Model 1 modeling data
# 2. fe_model2_congestion.parquet — Model 2 modeling data
# 3. analysis_model1.parquet — Model 1 analysis features
# 4. analysis_model2.parquet — Model 2 analysis features
# ============================================================

print("\n" + "=" * 70)
print("SAVING FEATURE-ENGINEERED DATASETS")
print("=" * 70)

# Save paths
save_dir = r"C:\Users\hunda\OneDrive\Desktop\Machine Learning Project\processed"

# Model 1 — Duration Regression
m1_path = f"{save_dir}\\fe_model1_duration.parquet"
df_m1.to_parquet(m1_path, index=False)
print(f"\n✅ Model 1 saved: {m1_path}")
print(f"   Shape: {df_m1.shape[0]:,} rows × {df_m1.shape[1]} columns")

# Model 2 — Congestion Classification
m2_path = f"{save_dir}\\fe_model2_congestion.parquet"
df_m2.to_parquet(m2_path, index=False)
print(f"\n✅ Model 2 saved: {m2_path}")
print(f"   Shape: {df_m2.shape[0]:,} rows × {df_m2.shape[1]} columns")

# Analysis — Model 1
a1_path = f"{save_dir}\\analysis_model1.parquet"
df_analysis_m1.to_parquet(a1_path, index=False)
print(f"\n✅ Analysis M1 saved: {a1_path}")
print(f"   Shape: {df_analysis_m1.shape[0]:,} rows × {df_analysis_m1.shape[1]} columns")

# Analysis — Model 2
a2_path = f"{save_dir}\\analysis_model2.parquet"
df_analysis_m2.to_parquet(a2_path, index=False)
print(f"\n✅ Analysis M2 saved: {a2_path}")
print(f"   Shape: {df_analysis_m2.shape[0]:,} rows × {df_analysis_m2.shape[1]} columns")


SAVING FEATURE-ENGINEERED DATASETS

✅ Model 1 saved: C:\Users\hunda\OneDrive\Desktop\Machine Learning Project\processed\fe_model1_duration.parquet
   Shape: 2,721,501 rows × 63 columns

✅ Model 2 saved: C:\Users\hunda\OneDrive\Desktop\Machine Learning Project\processed\fe_model2_congestion.parquet
   Shape: 2,412,426 rows × 60 columns

✅ Analysis M1 saved: C:\Users\hunda\OneDrive\Desktop\Machine Learning Project\processed\analysis_model1.parquet
   Shape: 2,721,501 rows × 9 columns

✅ Analysis M2 saved: C:\Users\hunda\OneDrive\Desktop\Machine Learning Project\processed\analysis_model2.parquet
   Shape: 2,412,426 rows × 9 columns


In [137]:
# ============================================================
# MODULE 5 COMPLETE — FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("MODULE 5 COMPLETE: FEATURE ENGINEERING")
print("=" * 70)

print(f"""
📊 JOURNEY SUMMARY:
   Raw data:       3,475,226 rows × 20 columns
   After cleaning: Model 1 = 2,721,501 × 22 | Model 2 = 2,412,426 × 22
   After features: Model 1 = 2,721,501 × 63 | Model 2 = 2,412,426 × 60

📋 FEATURES CREATED:
   Phase 1 (Temporal):     11 features
   Phase 2 (Geographic):   11 features
   Phase 3 (Trip):       10 features (4 analysis-only)
   Phase 4 (Encoding):     14 encoded cols + 2 interactions
   Phase 5 (All    — Cleanup):      Encoding + dropping
   Phase 6 (All    — Verify/Save):  Quality checks + save

📁 FILES SAVED:
   1. fe_model1_duration.parquet    → For duration regression modeling
   2. fe_model2_congestion.parquet  → For congestion fee classification
   3. analysis_model1.parquet       → Stakeholder analysis (Maria)
   4. analysis_model2.parquet       → Stakeholder analysis (Maria)

🎯 NEXT STEP: Module 6 — Baseline Models!
""")


MODULE 5 COMPLETE: FEATURE ENGINEERING

📊 JOURNEY SUMMARY:
   Raw data:       3,475,226 rows × 20 columns
   After cleaning: Model 1 = 2,721,501 × 22 | Model 2 = 2,412,426 × 22
   After features: Model 1 = 2,721,501 × 63 | Model 2 = 2,412,426 × 60

📋 FEATURES CREATED:
   Phase 1 (Temporal):     11 features
   Phase 2 (Geographic):   11 features
   Phase 3 (Trip):       10 features (4 analysis-only)
   Phase 4 (Encoding):     14 encoded cols + 2 interactions
   Phase 5 (All    — Cleanup):      Encoding + dropping
   Phase 6 (All    — Verify/Save):  Quality checks + save

📁 FILES SAVED:
   1. fe_model1_duration.parquet    → For duration regression modeling
   2. fe_model2_congestion.parquet  → For congestion fee classification
   3. analysis_model1.parquet       → Stakeholder analysis (Maria)
   4. analysis_model2.parquet       → Stakeholder analysis (Maria)

🎯 NEXT STEP: Module 6 — Baseline Models!

